In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace  = pd.read_csv('trace.csv')
df_log    = pd.read_csv('log.csv')
df_err    = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric
for df in (df_metric, df_trace, df_log, df_err):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# Helper to convert epoch seconds to timezone-aware datetime (Asia/Shanghai)
def to_dt_series(ts_series):
    # ts_series may have pandas nullable Int64; dropna for conversion then reindex
    s = ts_series.dropna().astype(int)
    dt = pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)
    return dt.reindex(ts_series.index)

# --- metric.csv aggregates ---
metric_pairs = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['count','cmdb_id','kpi_name'], ascending=[False, True, True])
)
metric_min_epoch = int(df_metric['timestamp'].min()) if not df_metric['timestamp'].isna().all() else None
metric_max_epoch = int(df_metric['timestamp'].max()) if not df_metric['timestamp'].isna().all() else None
metric_minmax = pd.DataFrame({
    'epoch': [metric_min_epoch, metric_max_epoch],
    'as_datetime_asia_shanghai': [
        pd.to_datetime(metric_min_epoch, unit='s', utc=True).tz_convert(tz) if metric_min_epoch is not None else pd.NaT,
        pd.to_datetime(metric_max_epoch, unit='s', utc=True).tz_convert(tz) if metric_max_epoch is not None else pd.NaT
    ],
    'which': ['min', 'max']
})[['which','epoch','as_datetime_asia_shanghai']]

# --- trace.csv aggregates ---
trace_pairs = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['count','cmdb_id','trace_name'], ascending=[False, True, True])
)
trace_min_epoch = int(df_trace['timestamp'].min()) if not df_trace['timestamp'].isna().all() else None
trace_max_epoch = int(df_trace['timestamp'].max()) if not df_trace['timestamp'].isna().all() else None
trace_minmax = pd.DataFrame({
    'epoch': [trace_min_epoch, trace_max_epoch],
    'as_datetime_asia_shanghai': [
        pd.to_datetime(trace_min_epoch, unit='s', utc=True).tz_convert(tz) if trace_min_epoch is not None else pd.NaT,
        pd.to_datetime(trace_max_epoch, unit='s', utc=True).tz_convert(tz) if trace_max_epoch is not None else pd.NaT
    ],
    'which': ['min', 'max']
})[['which','epoch','as_datetime_asia_shanghai']]

# --- log.csv aggregates ---
log_pairs = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['count','cmdb_id','log_name'], ascending=[False, True, True])
)
log_min_epoch = int(df_log['timestamp'].min()) if not df_log['timestamp'].isna().all() else None
log_max_epoch = int(df_log['timestamp'].max()) if not df_log['timestamp'].isna().all() else None
log_minmax = pd.DataFrame({
    'epoch': [log_min_epoch, log_max_epoch],
    'as_datetime_asia_shanghai': [
        pd.to_datetime(log_min_epoch, unit='s', utc=True).tz_convert(tz) if log_min_epoch is not None else pd.NaT,
        pd.to_datetime(log_max_epoch, unit='s', utc=True).tz_convert(tz) if log_max_epoch is not None else pd.NaT
    ],
    'which': ['min', 'max']
})[['which','epoch','as_datetime_asia_shanghai']]

# --- error_logs.csv aggregates ---
err_counts = (
    df_err
    .groupby(['cmdb_id'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['count','cmdb_id'], ascending=[False, True])
)
err_min_epoch = int(df_err['timestamp'].min()) if not df_err['timestamp'].isna().all() else None
err_max_epoch = int(df_err['timestamp'].max()) if not df_err['timestamp'].isna().all() else None
err_minmax = pd.DataFrame({
    'epoch': [err_min_epoch, err_max_epoch],
    'as_datetime_asia_shanghai': [
        pd.to_datetime(err_min_epoch, unit='s', utc=True).tz_convert(tz) if err_min_epoch is not None else pd.NaT,
        pd.to_datetime(err_max_epoch, unit='s', utc=True).tz_convert(tz) if err_max_epoch is not None else pd.NaT
    ],
    'which': ['min', 'max']
})[['which','epoch','as_datetime_asia_shanghai']]

# Up to 5 sample messages (choose earliest 5)
err_samples = df_err.sort_values('timestamp', ascending=True)[['timestamp','cmdb_id','message']].head(5).copy()
# Convert timestamp to Asia/Shanghai datetime for display
err_samples['as_datetime_asia_shanghai'] = to_dt_series(err_samples['timestamp'])

# Compact the outputs: limit rows displayed to at most 20 for pair lists
metric_pairs_display = metric_pairs.head(20)
trace_pairs_display  = trace_pairs.head(20)
log_pairs_display    = log_pairs.head(20)
err_counts_display   = err_counts.head(20)

# Final displayed variables (IPython will show them)
metric_pairs_display, metric_minmax, trace_pairs_display, trace_minmax, log_pairs_display, log_minmax, err_counts_display, err_minmax, err_samples

```
Out[1]:
```
```python
# Compose a concise plain-English summary string based on the previous execution results.
summary = (
    "Summary of telemetry files (based on executed aggregates):\n\n"
    "1) metric.csv\n"
    f"   - Distinct (cmdb_id, kpi_name) pairs: {metric_pairs.shape[0]}\n"
    f"   - Total metric rows: {len(df_metric)}\n"
    f"   - Timestamp range (Asia/Shanghai): min={metric_minmax.loc[metric_minmax['which']=='min','as_datetime_asia_shanghai'].iat[0]}, "
    f"max={metric_minmax.loc[metric_minmax['which']=='max','as_datetime_asia_shanghai'].iat[0]}\n"
    "   - Notes: Many pairs have 31 rows each (typical per-minute series over the interval).\n\n"
    "2) trace.csv\n"
    f"   - Distinct (cmdb_id, trace_name) pairs: {trace_pairs.shape[0]}\n"
    f"   - Total trace rows: {len(df_trace)}\n"
    f"   - Timestamp range (Asia/Shanghai): min={trace_minmax.loc[trace_minmax['which']=='min','as_datetime_asia_shanghai'].iat[0]}, "
    f"max={trace_minmax.loc[trace_minmax['which']=='max','as_datetime_asia_shanghai'].iat[0]}\n"
    "   - Notes: Most trace series show 31 samples per pair across the same interval.\n\n"
    "3) log.csv\n"
    f"   - Distinct (cmdb_id, log_name) pairs: {log_pairs.shape[0]}\n"
    f"   - Total log rows: {len(df_log)}\n"
    f"   - Timestamp range (Asia/Shanghai): min={log_minmax.loc[log_minmax['which']=='min','as_datetime_asia_shanghai'].iat[0]}, "
    f"max={log_minmax.loc[log_minmax['which']=='max','as_datetime_asia_shanghai'].iat[0]}\n"
    "   - Notes: Most log metrics have 31 rows per pair; some log.gc_pause_max series have fewer samples (e.g., 10–12 rows).\n\n"
    "4) error_logs.csv\n"
    f"   - Distinct cmdb_id entries: {err_counts.shape[0]}\n"
    f"   - Total error_logs rows: {len(df_err)}\n"
    "   - Timestamp range (Asia/Shanghai): "
    + (f"min={err_minmax.loc[err_minmax['which']=='min','as_datetime_asia_shanghai'].iat[0]}, "
       f"max={err_minmax.loc[err_minmax['which']=='max','as_datetime_asia_shanghai'].iat[0]}\n")
      if (not err_minmax['as_datetime_asia_shanghai'].isna().all()) else "no rows (empty)\n"
    "   - Sample messages: none (no error log rows were returned).\n\n"
    "Overall conclusion: metric, trace, and log files cover the same time window (2021-03-25 19:30:00+08 to 2021-03-25 20:00:00+08). "
    "Most per-(cmdb_id,metric) series contain 31 samples (per-minute data across the interval). error_logs.csv contains no entries in this dataset."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                     kpi_name  count
0      IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
2      IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
3      IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
6      IG01     JVM-Threads_7778_JVM_ThreadCount_Threads     31
88     IG02     JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
90     IG02    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
91     IG02  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
94     IG02     JVM-Threads_7778_JVM_ThreadCount_Threads     31
176    MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31
178    MG01    JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31
179    MG01  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     31
182    MG01     JVM-Threads_7779_JVM_ThreadCount_Threads     31
281    MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31
283    MG02    JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31
284    MG02  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     31
287    MG02     JVM-Threads_7779_JVM_ThreadCount_Threads     31
297    MG02                      OSLinux-CPU_CPU_CPULoad     31
298    MG02                   OSLinux-CPU_CPU_CPUSysTime     31
299    MG02                  OSLinux-CPU_CPU_CPUUserTime     31
300    MG02                       OSLinux-CPU_CPU_CPUWio     31,   which       epoch as_datetime_asia_shanghai
0   min  1616671800 2021-03-25 19:30:00+08:00
1   max  1616673600 2021-03-25 20:00:00+08:00,    cmdb_id                       trace_name  count
3     IG01         trace.self.duration_mean     31
4     IG01          trace.self.duration_p95     31
5     IG01             trace.self.row_count     31
6     IG01  trace.to_Tomcat01.duration_mean     31
7     IG01   trace.to_Tomcat01.duration_p95     31
8     IG01      trace.to_Tomcat01.row_count     31
9     IG01  trace.to_Tomcat02.duration_mean     31
10    IG01   trace.to_Tomcat02.duration_p95     31
11    IG01      trace.to_Tomcat02.row_count     31
12    IG01  trace.to_Tomcat03.duration_mean     31
13    IG01   trace.to_Tomcat03.duration_p95     31
14    IG01      trace.to_Tomcat03.row_count     31
15    IG01  trace.to_Tomcat04.duration_mean     31
16    IG01   trace.to_Tomcat04.duration_p95     31
17    IG01      trace.to_Tomcat04.row_count     31
21    IG02         trace.self.duration_mean     31
22    IG02          trace.self.duration_p95     31
23    IG02             trace.self.row_count     31
24    IG02  trace.to_Tomcat01.duration_mean     31
25    IG02   trace.to_Tomcat01.duration_p95     31,   which       epoch as_datetime_asia_shanghai
0   min  1616671800 2021-03-25 19:30:00+08:00
1   max  1616673600 2021-03-25 20:00:00+08:00,      cmdb_id          log_name  count
0   Tomcat01   log.error_count     31
2   Tomcat01     log.row_count     31
3   Tomcat02   log.error_count     31
5   Tomcat02     log.row_count     31
6   Tomcat03   log.error_count     31
8   Tomcat03     log.row_count     31
9   Tomcat04   log.error_count     31
11  Tomcat04     log.row_count     31
12  apache01   log.error_count     31
13  apache01     log.row_count     31
14  apache02   log.error_count     31
15  apache02     log.row_count     31
10  Tomcat04  log.gc_pause_max     12
1   Tomcat01  log.gc_pause_max     10
4   Tomcat02  log.gc_pause_max     10
7   Tomcat03  log.gc_pause_max     10,   which       epoch as_datetime_asia_shanghai
0   min  1616671800 2021-03-25 19:30:00+08:00
1   max  1616673600 2021-03-25 20:00:00+08:00, Empty DataFrame
Columns: [cmdb_id, count]
Index: [],   which epoch as_datetime_asia_shanghai
0   min  None                       NaT
1   max  None                       NaT, Empty DataFrame
Columns: [timestamp, cmdb_id, message, as_datetime_asia_shanghai]
Index: [])```
```

In [ ]:
# In[2]:

# Reuse existing loaded DataFrames: df_metric, df_trace, df_log
import pandas as pd
# Ensure numeric values for threshold calculations
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value']  = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value']    = pd.to_numeric(df_log['value'], errors='coerce')

# Helper to build threshold table for a given df and name column
def build_thresholds(df, name_col):
    # aggregate: count of non-null points, p95, p50, p5 computed on full series per group
    grouped = (
        df
        .groupby(['cmdb_id', name_col], dropna=False)['value']
        .agg(count=('count'), 
             p95=lambda x: x.quantile(0.95),
             p50=lambda x: x.quantile(0.50),
             p5 =lambda x: x.quantile(0.05))
        .reset_index()
    )
    # rename the metric column to a common name 'metric_name'
    grouped = grouped.rename(columns={name_col: 'metric_name'})
    # sort by descending p95 for prioritization
    grouped = grouped.sort_values('p95', ascending=False).reset_index(drop=True)
    return grouped

# Build full threshold tables and store in kernel variables as required
thresh_metric = build_thresholds(df_metric, 'kpi_name')
thresh_trace  = build_thresholds(df_trace,  'trace_name')
thresh_log    = build_thresholds(df_log,    'log_name')

# Prepare compact displays: top 20 rows each, show selected columns and round numeric thresholds
top_metric_display = thresh_metric.head(20).copy()
top_trace_display  = thresh_trace.head(20).copy()
top_log_display    = thresh_log.head(20).copy()

for df in (top_metric_display, top_trace_display, top_log_display):
    df['p95'] = df['p95'].round(6)
    df['p50'] = df['p50'].round(6)
    df['p5']  = df['p5'].round(6)

# Display the three compact tables (at most 20 rows each)
top_metric_display[['cmdb_id','metric_name','count','p95','p50','p5']], \
top_trace_display[['cmdb_id','metric_name','count','p95','p50','p5']], \
top_log_display[['cmdb_id','metric_name','count','p95','p50','p5']]

```
Out[2]:
```
# Build a concise plain-English summary using the previously computed threshold tables:
def fmt_num(v):
    import pandas as _pd
    if _pd.isna(v):
        return "NA"
    try:
        return f"{v:.6g}"
    except Exception:
        return str(v)

topn = 5
m_top = thresh_metric.head(topn)[['cmdb_id','metric_name','count','p95']].copy()
t_top = thresh_trace.head(topn)[['cmdb_id','metric_name','count','p95']].copy()
l_top = thresh_log.head(topn)[['cmdb_id','metric_name','count','p95']].copy()

lines = []
lines.append("Summary of per-series global thresholds (top groups by P95):\n")

lines.append("Metrics (top 5 by P95):")
for i, row in enumerate(m_top.itertuples(index=False), 1):
    lines.append(f" {i}) {row.cmdb_id} - {row.metric_name} | count={int(row.count)} | P95={fmt_num(row.p95)}")

lines.append("\nTraces (top 5 by P95):")
for i, row in enumerate(t_top.itertuples(index=False), 1):
    lines.append(f" {i}) {row.cmdb_id} - {row.metric_name} | count={int(row.count)} | P95={fmt_num(row.p95)}")

lines.append("\nLogs (top 5 by P95):")
for i, row in enumerate(l_top.itertuples(index=False), 1):
    lines.append(f" {i}) {row.cmdb_id} - {row.metric_name} | count={int(row.count)} | P95={fmt_num(row.p95)}")

lines.append("\nNotes:")
lines.append(" - Metric top groups are dominated by Mysql02 (very large numeric P95 values for IO/rows/bytes) and Redis entries.")
lines.append(" - Trace top groups are primarily trace.self.row_count for docker/ Tomcat instances (high request/row counts).")
lines.append(" - Log top groups show apache/apache access row counts highest; GC pause metrics are small decimals; error_count is zero for top entries.")
lines.append(" - Full per-group threshold tables are stored in kernel variables: thresh_metric, thresh_trace, thresh_log.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                        metric_name  count           p95           p50            p5
0    Mysql02               Mysql-MySQL_3306_Innodb data written     30  2.849051e+12  5.264384e+05  2.150921e+05
1    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests     30  1.665867e+12  4.012469e+03  2.103123e+03
2    Mysql02             Mysql-MySQL_3306_Innodb os log written     30  7.538172e+11  2.893533e+05  1.610675e+05
3    Mysql01               Mysql-MySQL_3306_Innodb data written     30  6.449695e+11  1.837397e+05  1.025536e+05
4    Mysql02                         Mysql-MySQL_3306_Rows Read     30  5.445324e+11  7.518000e+02  1.782700e+02
5    Mysql02                 Mysql-MySQL_3306_Handler Read Prev     30  5.443370e+11  6.964750e+02  1.279298e+02
6    Mysql02                        Mysql-MySQL_3306_Bytes Sent     30  1.937478e+11  6.900687e+04  4.259392e+04
7    Mysql01                        Mysql-MySQL_3306_Bytes Sent     30  8.047871e+10  1.893024e+04  1.195182e+04
8    Mysql02                  Mysql-MySQL_3306_Innodb data read     30  3.317313e+10  0.000000e+00  0.000000e+00
9    Mysql01             Mysql-MySQL_3306_Innodb os log written     30  9.442049e+09  2.468664e+03  1.564124e+03
10   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     22  7.113537e+09  7.112717e+09  7.111932e+09
11   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     22  6.334105e+09  6.332388e+09  6.330858e+09
12   Redis01            redis-Redis_6379_Redis  (keyspace_hits)     22  5.451805e+09  5.451182e+09  5.450590e+09
13   Redis02            redis-Redis_6379_Redis  (keyspace_hits)     22  4.683676e+09  4.682369e+09  4.681210e+09
14   Redis01                redis-Redis_6379_Redis  (maxmemory)     22  4.294967e+09  4.294967e+09  4.294967e+09
15   Redis02                redis-Redis_6379_Redis  (maxmemory)     22  4.294967e+09  4.294967e+09  4.294967e+09
16  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09
17  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09
18  Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09
19  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09,      cmdb_id                    metric_name  count     p95     p50      p5
0   dockerB1           trace.self.row_count     31  3132.5  2229.0  1105.0
1   Tomcat03           trace.self.row_count     31  1807.0  1389.0  1140.0
2   Tomcat01           trace.self.row_count     31  1787.0  1374.0  1117.5
3   Tomcat04           trace.self.row_count     31  1745.5  1430.0  1076.5
4   Tomcat02           trace.self.row_count     31  1616.0  1418.0   987.0
5   dockerA2           trace.self.row_count     31  1393.5   988.0   535.0
6   dockerA1           trace.self.row_count     31  1364.0   976.0   542.0
7       IG01           trace.self.row_count     31  1028.0   816.0   705.0
8   dockerB2           trace.self.row_count     31  1026.0   689.0   396.5
9       IG02           trace.self.row_count     31  1005.0   814.0   646.0
10      MG02           trace.self.row_count     31   790.0   624.0   508.5
11      MG01           trace.self.row_count     31   748.0   580.0   513.0
12      MG02  trace.from_dockerB1.row_count     31   217.0   129.0    56.5
13  dockerB1        trace.to_MG02.row_count     31   217.0   129.0    56.5
14      MG01    trace.to_dockerB1.row_count     31   201.5   146.0   107.0
15  dockerB1      trace.from_MG01.row_count     31   201.5   146.0   107.0
16  dockerB1      trace.from_MG02.row_count     31   197.0   147.0   102.5
17      MG02    trace.to_dockerB1.row_count     31   197.0   147.0   102.5
18      MG01  trace.from_dockerB1.row_count     31   185.5   122.0    47.5
19  dockerB1        trace.to_MG01.row_count     31   185.5   122.0    47.5,      cmdb_id       metric_name  count       p95      p50        p5
0   apache02     log.row_count     31  511.5000  418.000  341.5000
1   apache01     log.row_count     31  495.5000  405.000  340.5000
2   Tomcat03     log.row_count     31  268.0000  210.000  177.0000
3   Tomcat01     log.row_count     31  265.0000  204.000  174.5000
4   Tomcat04     log.row_count     31  258.0000  217.000  163.0000
5   Tomcat02     log.row_count     31  243.0000  206.000  145.5000
6   Tomcat04  log.gc_pause_max     12    0.1100    0.070    0.0355
7   Tomcat02  log.gc_pause_max     10    0.0910    0.070    0.0300
8   Tomcat01  log.gc_pause_max     10    0.0855    0.065    0.0245
9   Tomcat03  log.gc_pause_max     10    0.0800    0.070    0.0435
10  Tomcat01   log.error_count     31    0.0000    0.000    0.0000
11  Tomcat02   log.error_count     31    0.0000    0.000    0.0000
12  Tomcat03   log.error_count     31    0.0000    0.000    0.0000
13  Tomcat04   log.error_count     31    0.0000    0.000    0.0000
14  apache01   log.error_count     31    0.0000    0.000    0.0000
15  apache02   log.error_count     31    0.0000    0.000    0.0000)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes: df_metric, thresh_metric (thresh_metric has columns: cmdb_id, metric_name, count, p95, p50, p5)

# Incident window (inclusive)
start_ts = 1616671800
end_ts   = 1616673600

# Filter metric rows in the window
df_metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Ensure numeric value
df_metric_window['value'] = pd.to_numeric(df_metric_window['value'], errors='coerce')

# Per-group summary within window
group_win = (
    df_metric_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        count_in_window = ('value','count'),
        earliest_ts = ('timestamp','min'),
        min_value_in_window = ('value','min'),
        max_value_in_window = ('value','max')
    )
    .reset_index()
)

# Merge global thresholds from thresh_metric (metric_name corresponds to kpi_name)
# Ensure thresh_metric exists and has numeric p95/p50/p5
thresh_metric['p95'] = pd.to_numeric(thresh_metric['p95'], errors='coerce')
thresh_metric['p50'] = pd.to_numeric(thresh_metric['p50'], errors='coerce')
thresh_metric['p5']  = pd.to_numeric(thresh_metric['p5'], errors='coerce')

group_win = group_win.merge(
    thresh_metric.rename(columns={'metric_name':'kpi_name'}),
    on=['cmdb_id','kpi_name'],
    how='left',
    suffixes=('','_global')
)

# Determine anomaly existence per group: max>p95 OR min<p5
group_win['anomaly'] = (
    (group_win['max_value_in_window'] > group_win['p95']) |
    (group_win['min_value_in_window'] < group_win['p5'])
)

# To get earliest_anomaly_timestamp per group, find row-level anomalies by merging per-row thresholds
# Merge row-level thresholds by matching (cmdb_id,kpi_name)
rows_with_thresh = df_metric_window.merge(
    thresh_metric.rename(columns={'metric_name':'kpi_name'}),
    on=['cmdb_id','kpi_name'],
    how='left'
)
# Flag anomalous rows
rows_with_thresh['is_anom_row'] = (rows_with_thresh['value'] > rows_with_thresh['p95']) | (rows_with_thresh['value'] < rows_with_thresh['p5'])
# Earliest anomaly timestamp per group
earliest_anom = (
    rows_with_thresh[rows_with_thresh['is_anom_row']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_ts'})
)

# Join earliest anomaly ts into group_win
group_win = group_win.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Compute severity_score per spec:
# severity = max(max_value_in_window/p95, p5/min_value_in_window) with safe handling for zeros/NaN
SENTINEL = 1e18

def safe_div(numer, denom):
    try:
        if pd.isna(numer) or pd.isna(denom):
            return float('nan')
        denom = float(denom)
        if denom == 0:
            return SENTINEL
        return float(numer) / denom
    except Exception:
        return SENTINEL

group_win['ratio_up'] = group_win.apply(lambda r: safe_div(r['max_value_in_window'], r['p95']), axis=1)
group_win['ratio_down'] = group_win.apply(lambda r: safe_div(r['p5'], r['min_value_in_window']), axis=1)
# Use the part that applies; overall severity is max of the two ratios (nan-safe)
group_win['severity_score'] = group_win[['ratio_up','ratio_down']].max(axis=1)

# Filter only anomalous groups
anomalies_metric = group_win[group_win['anomaly']].copy()

# Convert earliest_anomaly_ts and earliest_ts to Asia/Shanghai datetimes for display
def epoch_to_as_tz(epoch):
    if pd.isna(epoch):
        return pd.NaT
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz)

anomalies_metric['earliest_anomaly_datetime_asia_shanghai'] = anomalies_metric['earliest_anomaly_ts'].apply(epoch_to_as_tz)
anomalies_metric['earliest_window_datetime_asia_shanghai'] = anomalies_metric['earliest_ts'].apply(epoch_to_as_tz)

# Select and order columns for final compact table
out_cols = [
    'cmdb_id',
    'kpi_name',
    'count_in_window',
    'earliest_anomaly_datetime_asia_shanghai',
    'min_value_in_window',
    'max_value_in_window',
    'p95','p50','p5',
    'severity_score'
]

# Prepare compact top-20 sorted by descending severity_score
anomalies_metric_top = (
    anomalies_metric
    .sort_values('severity_score', ascending=False)
    .reset_index(drop=True)
    [out_cols]
    .head(20)
    .copy()
)

# Round numeric columns for compactness
for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score']:
    if c in anomalies_metric_top.columns:
        anomalies_metric_top[c] = pd.to_numeric(anomalies_metric_top[c], errors='coerce').round(6)

# Display the compact table (up to 20 rows)
anomalies_metric_top

```
Out[3]:
```
# Summarize the anomaly detection results in plain English using existing kernel variables.
# Reuse anomalies_metric and anomalies_metric_top computed earlier.
SENTINEL = 1e18

total_anomalous_groups = int(anomalies_metric.shape[0]) if 'anomalies_metric' in globals() else 0
num_sentinel_severity = int(anomalies_metric[anomalies_metric['severity_score'] >= SENTINEL].shape[0]) if 'anomalies_metric' in globals() else 0

# Prepare top 5 anomalous groups for brief examples
if 'anomalies_metric_top' in globals():
    top5 = anomalies_metric_top.head(5)[['cmdb_id','kpi_name','earliest_anomaly_datetime_asia_shanghai','severity_score']].copy()
else:
    top5 = None

lines = []
lines.append("Anomaly detection summary for metric series within window 1616671800–1616673600 (Asia/Shanghai):")
lines.append(f"- Total anomalous (cmdb_id, kpi_name) groups found: {total_anomalous_groups}")
lines.append(f"- Groups with sentinel severity (due to division-by-zero from min_value_in_window == 0): {num_sentinel_severity}")
lines.append("")
lines.append("Top anomalous groups (examples):")
if top5 is None or top5.shape[0] == 0:
    lines.append("  (no example rows available)")
else:
    for i, r in enumerate(top5.itertuples(index=False), 1):
        ts = r.earliest_anomaly_datetime_asia_shanghai
        ts_str = str(ts) if pd.notna(ts) else "N/A"
        sev = r.severity_score
        sev_str = f"{sev:.6g}" if pd.notna(sev) else "NA"
        lines.append(f" {i}) {r.cmdb_id} | {r.kpi_name} | earliest_anomaly={ts_str} | severity={sev_str}")

lines.append("")
lines.append("Key observations:")
lines.append("- Many anomalies are for OSLinux disk and CPU-related metrics where min_value_in_window == 0,")
lines.append("  causing a sentinel severity due to safe division handling (investigate zero readings).")
lines.append("- Several disk throughput/IO metrics (DSKBps/DSKRead/DSKRTps/DSKPer..., on apache, IG02, Tomcat nodes) have anomalies.")
lines.append("- Actionable next steps: inspect node disk health and metrics ingestion (zero values), check disk I/O and CPU idle readings, and confirm no missing samples or agent issues.")

summary = "\n".join(lines)
summary, top5

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window earliest_anomaly_datetime_asia_shanghai  min_value_in_window  max_value_in_window         p95        p50         p5  severity_score
0   apache01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                9               2021-03-25 19:46:00+08:00                  0.0             3.000000    1.800000   0.000000   0.000000    1.000000e+18
1   apache01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                9               2021-03-25 19:52:00+08:00                  0.0             0.400000    0.240000   0.000000   0.000000    1.000000e+18
2   apache01                    OSLinux-CPU_CPU-1_SingleCpuidle                8               2021-03-25 19:37:00+08:00                  0.0            99.784100   99.760895  88.552100  11.385885    1.000000e+18
3       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               11               2021-03-25 19:54:00+08:00                  0.0             7.216700    3.700000   0.000000   0.000000    1.000000e+18
4   apache01                    OSLinux-CPU_CPU-3_SingleCpuidle                8               2021-03-25 19:47:00+08:00                  0.0            99.750900   99.733435  81.147350   0.000000    1.000000e+18
5       IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               11               2021-03-25 19:50:00+08:00                  0.0             3.000000    1.500000   0.000000   0.000000    1.000000e+18
6       IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...               10               2021-03-25 19:54:00+08:00                  0.0            17.800000    9.925000   0.000000   0.000000    1.000000e+18
7       IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               10               2021-03-25 19:51:00+08:00                  0.0             1.000000    0.550000   0.000000   0.000000    1.000000e+18
8       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               11               2021-03-25 19:54:00+08:00                  0.0           626.800000  316.400000   0.000000   0.000000    1.000000e+18
9   Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...               16               2021-03-25 19:52:00+08:00                  0.0            10.400000    2.825000   0.000000   0.000000    1.000000e+18
10      IG02                    OSLinux-CPU_CPU-0_SingleCpuidle               11               2021-03-25 19:40:00+08:00                  0.0            99.002300   98.969250   0.000000   0.000000    1.000000e+18
11  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               15               2021-03-25 19:52:00+08:00                  0.0             6.783333    2.280000   0.000000   0.000000    1.000000e+18
12  Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle               15               2021-03-25 19:59:00+08:00                  0.0            97.189400   97.186460   0.797077   0.000000    1.000000e+18
13  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...               15               2021-03-25 19:42:00+08:00                  0.0             0.400000    0.330000   0.000000   0.000000    1.000000e+18
14  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               16               2021-03-25 19:30:00+08:00                  0.0             8.000000    4.250000   0.000000   0.000000    1.000000e+18
15  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               11               2021-03-25 19:53:00+08:00                  0.0             7.000000    4.000000   0.000000   0.000000    1.000000e+18
16  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               15               2021-03-25 19:52:00+08:00                  0.0           438.800000  186.146690   0.000000   0.000000    1.000000e+18
17  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               16               2021-03-25 19:44:00+08:00                  0.0           381.733333  125.433333   0.000000   0.000000    1.000000e+18
18  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               11               2021-03-25 19:45:00+08:00                  0.0             9.000000    5.000000   0.000000   0.000000    1.000000e+18
19      IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               10               2021-03-25 19:54:00+08:00                  0.0             6.000000    4.200000   1.000000   0.000000    1.000000e+18```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# 1) Per-component summary from anomalies_metric (assumed present in kernel)
# Use earliest_anomaly_ts if present, else try datetime field
if 'anomalies_metric' not in globals():
    raise RuntimeError("anomalies_metric not found in kernel. Run previous metric anomaly step first.")

am = anomalies_metric.copy()

# ensure earliest anomaly epoch column exists
if 'earliest_anomaly_ts' in am.columns:
    am['earliest_anomaly_epoch'] = am['earliest_anomaly_ts']
elif 'earliest_anomaly_datetime_asia_shanghai' in am.columns:
    # convert datetime back to epoch if possible
    am['earliest_anomaly_epoch'] = am['earliest_anomaly_datetime_asia_shanghai'].apply(
        lambda d: int(d.tz_convert('UTC').timestamp()) if pd.notna(d) else pd.NA
    )
else:
    am['earliest_anomaly_epoch'] = pd.NA

# Group per component
comp_summary = (
    am
    .groupby('cmdb_id', dropna=False)
    .agg(
        number_of_anomalous_metric_groups = ('kpi_name','nunique'),
        earliest_anomaly_epoch = ('earliest_anomaly_epoch','min'),
        max_severity_score = ('severity_score','max')
    )
    .reset_index()
)

# Convert earliest epoch to Asia/Shanghai datetime for display
def epoch_to_asia_shanghai(e):
    if pd.isna(e):
        return pd.NaT
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz)

comp_summary['earliest_anomaly_datetime_asia_shanghai'] = comp_summary['earliest_anomaly_epoch'].apply(epoch_to_asia_shanghai)
# Round severity
comp_summary['max_severity_score'] = pd.to_numeric(comp_summary['max_severity_score'], errors='coerce').round(6)

# Select and sort top 20 by number_of_anomalous_metric_groups desc
comp_summary_top = (
    comp_summary
    .sort_values(['number_of_anomalous_metric_groups','max_severity_score'], ascending=[False, False])
    .reset_index(drop=True)
    [['cmdb_id','number_of_anomalous_metric_groups','earliest_anomaly_datetime_asia_shanghai','max_severity_score']]
    .head(20)
)

# 2) Trace anomaly detection within the same window using thresh_trace (assumed present)

# Validate presence
if 'thresh_trace' not in globals():
    raise RuntimeError("thresh_trace not found in kernel. Compute global thresholds for traces first.")

# Incident window
start_ts = 1616671800
end_ts   = 1616673600

# Reuse df_trace loaded previously
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
df_trace_window['value'] = pd.to_numeric(df_trace_window['value'], errors='coerce')

# Per-group window summary
trace_group_win = (
    df_trace_window
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_in_window = ('value','count'),
        earliest_ts = ('timestamp','min'),
        min_value_in_window = ('value','min'),
        max_value_in_window = ('value','max')
    )
    .reset_index()
)

# Prepare thresh_trace: ensure numeric and trace metric name column 'metric_name'
thresh_trace['p95'] = pd.to_numeric(thresh_trace['p95'], errors='coerce')
thresh_trace['p50'] = pd.to_numeric(thresh_trace['p50'], errors='coerce')
thresh_trace['p5']  = pd.to_numeric(thresh_trace['p5'], errors='coerce')

# Merge global thresholds into group summary (match trace_name -> metric_name)
trace_group_win = trace_group_win.merge(
    thresh_trace.rename(columns={'metric_name':'trace_name'}),
    on=['cmdb_id','trace_name'],
    how='left'
)

# Flag anomalous groups by comparing window min/max to global thresholds
trace_group_win['anomaly'] = (
    (trace_group_win['max_value_in_window'] > trace_group_win['p95']) |
    (trace_group_win['min_value_in_window'] < trace_group_win['p5'])
)

# To get earliest anomaly timestamp per group, compute row-level anomalies
rows_with_thresh_t = df_trace_window.merge(
    thresh_trace.rename(columns={'metric_name':'trace_name'}),
    on=['cmdb_id','trace_name'],
    how='left'
)
rows_with_thresh_t['is_anom_row'] = (rows_with_thresh_t['value'] > rows_with_thresh_t['p95']) | (rows_with_thresh_t['value'] < rows_with_thresh_t['p5'])

earliest_anom_trace = (
    rows_with_thresh_t[rows_with_thresh_t['is_anom_row']]
    .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_ts'})
)

# Join earliest anomaly ts
trace_group_win = trace_group_win.merge(earliest_anom_trace, on=['cmdb_id','trace_name'], how='left')

# Severity calculation with safe division
def safe_div(numer, denom):
    if pd.isna(numer) or pd.isna(denom):
        return float('nan')
    denom = float(denom)
    if denom == 0:
        return SENTINEL
    return float(numer) / denom

trace_group_win['ratio_up'] = trace_group_win.apply(lambda r: safe_div(r['max_value_in_window'], r['p95']), axis=1)
trace_group_win['ratio_down'] = trace_group_win.apply(lambda r: safe_div(r['p5'], r['min_value_in_window']), axis=1)
trace_group_win['severity_score'] = trace_group_win[['ratio_up','ratio_down']].max(axis=1)

# Filter only anomalous groups
anomalies_trace = trace_group_win[trace_group_win['anomaly']].copy()

# Convert timestamps to Asia/Shanghai datetimes for display
anomalies_trace['earliest_anomaly_datetime_asia_shanghai'] = anomalies_trace['earliest_anomaly_ts'].apply(lambda e: epoch_to_asia_shanghai(e))
anomalies_trace['earliest_window_datetime_asia_shanghai'] = anomalies_trace['earliest_ts'].apply(lambda e: epoch_to_asia_shanghai(e))

# Prepare compact top-20 sorted by severity_score desc
anomalies_trace_top = (
    anomalies_trace
    .sort_values('severity_score', ascending=False)
    .reset_index(drop=True)
    [['cmdb_id','trace_name','count_in_window','earliest_anomaly_datetime_asia_shanghai',
      'min_value_in_window','max_value_in_window','p95','p50','p5','severity_score']]
    .head(20)
    .copy()
)

# Round numeric columns for compactness
for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score']:
    if c in anomalies_trace_top.columns:
        anomalies_trace_top[c] = pd.to_numeric(anomalies_trace_top[c], errors='coerce').round(6)

# Store full anomalies_trace in kernel (already assigned). anomalies_metric already stored earlier.
# Final displayed variables (compact summaries)
comp_summary_top, anomalies_trace_top

```
Out[4]:
```
```python
# Compose a concise plain-English summary string based on the latest anomaly results in kernel variables.
import pandas as pd

# Safety checks for required variables
required = ['comp_summary_top','anomalies_trace','anomalies_trace_top','anomalies_metric']
missing = [v for v in required if v not in globals()]
if missing:
    summary = "Required result variables missing from kernel: " + ", ".join(missing)
else:
    # Build metric-component summary (top components already in comp_summary_top)
    comp_df = comp_summary_top.copy()
    n_components_reported = comp_df.shape[0]
    total_anomalous_metric_groups = int(anomalies_metric.shape[0])
    # Top 5 components lines
    top_comp_lines = []
    for i, r in enumerate(comp_df.head(5).itertuples(index=False), 1):
        ts = r.earliest_anomaly_datetime_asia_shanghai
        ts_str = str(ts) if pd.notna(ts) else "N/A"
        top_comp_lines.append(f" {i}) {r.cmdb_id}: {r.number_of_anomalous_metric_groups} anomalous metric groups, earliest_anomaly={ts_str}, max_severity={r.max_severity_score:.6g}")

    # Trace anomalies summary
    trace_df = anomalies_trace.copy()
    n_trace_anoms = int(trace_df.shape[0])
    trace_top_lines = []
    for i, r in enumerate(anomalies_trace_top.head(5).itertuples(index=False), 1):
        ts = r.earliest_anomaly_datetime_asia_shanghai
        ts_str = str(ts) if pd.notna(ts) else "N/A"
        trace_top_lines.append(f" {i}) {r.cmdb_id} - {r.trace_name}: count_in_window={int(r.count_in_window)}, earliest_anomaly={ts_str}, severity={r.severity_score:.6g}")

    # Note about sentinel severity
    sentinel_note = ""
    if (anomalies_metric['severity_score'] >= 1e18).any():
        sentinel_note = ("Many metric anomalies show a very large sentinel severity (1e18) caused by zero min values "
                         "in-window (division-by-zero handled by sentinel). Investigate zero readings / ingestion issues.")

    # Assemble summary
    lines = [
        "Summary of anomaly results (window 1616671800–1616673600, Asia/Shanghai):",
        "",
        f"Metrics (per-component): {total_anomalous_metric_groups} anomalous (cmdb_id, kpi_name) groups across components.",
        f"Top {min(5, n_components_reported)} components by number of anomalous metric groups:",
    ]
    lines += top_comp_lines
    lines += ["", f"Displayed {n_components_reported} components (top 20)."]
    lines += ["", "Trace anomalies (per-series):", f"Total anomalous trace groups found: {n_trace_anoms}"]
    lines += ["Top trace anomalies (by severity):"]
    lines += trace_top_lines
    if sentinel_note:
        lines += ["", "Note:", sentinel_note]
    lines += [
        "",
        "Key takeaways:",
        "- Metric anomalies are widespread; top affected components include Mysql02, Mysql01, Tomcat03, MG02, MG01, Tomcat02, Tomcat04, Tomcat01, IG02, IG01.",
        "- Many metric anomalies are related to OS/disk/CPU series where min values hit 0 during the window (causing sentinel severity).",
        "- Trace anomalies are concentrated on high `trace.self.row_count` and several inter-service row_count traces (Tomcat and docker nodes).",
        "- Recommended next steps: inspect disk I/O and agent health on nodes with sentinel severities, and investigate traffic spikes for the top trace series (Tomcat02, Tomcat03, IG02, etc.)."
    ]
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  number_of_anomalous_metric_groups earliest_anomaly_datetime_asia_shanghai  max_severity_score
0         Mysql02                                 97               2021-03-25 19:30:00+08:00        1.000000e+18
1         Mysql01                                 92               2021-03-25 19:30:00+08:00        1.000000e+18
2        Tomcat03                                 50               2021-03-25 19:30:00+08:00        1.000000e+18
3            MG02                                 49               2021-03-25 19:30:00+08:00        1.000000e+18
4            MG01                                 48               2021-03-25 19:30:00+08:00        1.000000e+18
5        Tomcat02                                 48               2021-03-25 19:30:00+08:00        1.000000e+18
6        Tomcat04                                 47               2021-03-25 19:30:00+08:00        1.000000e+18
7        Tomcat01                                 43               2021-03-25 19:30:00+08:00        1.000000e+18
8            IG02                                 40               2021-03-25 19:30:00+08:00        1.000000e+18
9            IG01                                 39               2021-03-25 19:30:00+08:00        1.000000e+18
10        Redis02                                 36               2021-03-25 19:30:00+08:00        1.000000e+18
11        Redis01                                 35               2021-03-25 19:30:00+08:00        1.000000e+18
12       apache01                                 32               2021-03-25 19:30:00+08:00        1.000000e+18
13       apache02                                 28               2021-03-25 19:31:00+08:00        1.000000e+18
14       dockerA2                                  4               2021-03-25 19:32:00+08:00        1.245000e+00
15  ServiceTest11                                  2               2021-03-25 19:35:00+08:00        1.050000e+01
16   ServiceTest9                                  2               2021-03-25 19:35:00+08:00        6.083333e+00
17   ServiceTest8                                  2               2021-03-25 19:31:00+08:00        3.916667e+00
18   ServiceTest3                                  2               2021-03-25 19:31:00+08:00        2.792957e+00
19   ServiceTest5                                  2               2021-03-25 19:44:00+08:00        2.021784e+00,      cmdb_id                     trace_name  count_in_window earliest_anomaly_datetime_asia_shanghai  min_value_in_window  max_value_in_window     p95     p50      p5  severity_score
0   Tomcat02           trace.self.row_count               31               2021-03-25 19:37:00+08:00                  6.0               1681.0  1616.0  1418.0   987.0        164.5000
1   dockerB2           trace.self.row_count               31               2021-03-25 19:36:00+08:00                  4.0               1106.0  1026.0   689.0   396.5         99.1250
2       IG02           trace.self.row_count               31               2021-03-25 19:32:00+08:00                  8.0               1062.0  1005.0   814.0   646.0         80.7500
3   Tomcat03           trace.self.row_count               31               2021-03-25 19:37:00+08:00                 15.0               2021.0  1807.0  1389.0  1140.0         76.0000
4   Tomcat03        trace.to_MG02.row_count               31               2021-03-25 19:37:00+08:00                  1.0                139.0   136.0   102.0    75.0         75.0000
5       MG02  trace.from_Tomcat03.row_count               31               2021-03-25 19:37:00+08:00                  1.0                139.0   136.0   102.0    75.0         75.0000
6       MG02  trace.from_Tomcat02.row_count               31               2021-03-25 19:37:00+08:00                  1.0                131.0   126.0   105.0    74.0         74.0000
7   Tomcat02        trace.to_MG02.row_count               31               2021-03-25 19:37:00+08:00                  1.0                131.0   126.0   105.0    74.0         74.0000
8   Tomcat02      trace.from_IG01.row_count               31               2021-03-25 19:38:00+08:00                  1.0                137.0   127.0   103.0    73.0         73.0000
9       IG01    trace.to_Tomcat02.row_count               31               2021-03-25 19:38:00+08:00                  1.0                137.0   127.0   103.0    73.0         73.0000
10      MG02           trace.self.row_count               31               2021-03-25 19:31:00+08:00                  9.0                800.0   790.0   624.0   508.5         56.5000
11  dockerB1      trace.from_MG01.row_count               31               2021-03-25 19:38:00+08:00                  2.0                222.0   201.5   146.0   107.0         53.5000
12      MG01    trace.to_dockerB1.row_count               31               2021-03-25 19:38:00+08:00                  2.0                222.0   201.5   146.0   107.0         53.5000
13      MG01           trace.self.row_count               31               2021-03-25 19:32:00+08:00                 10.0                752.0   748.0   580.0   513.0         51.3000
14      MG02    trace.to_dockerB1.row_count               31               2021-03-25 19:38:00+08:00                  2.0                240.0   197.0   147.0   102.5         51.2500
15  dockerB1      trace.from_MG02.row_count               31               2021-03-25 19:38:00+08:00                  2.0                240.0   197.0   147.0   102.5         51.2500
16  Tomcat04      trace.from_IG01.row_count               31               2021-03-25 19:45:00+08:00                  2.0                153.0   130.0   106.0    88.5         44.2500
17      IG01    trace.to_Tomcat04.row_count               31               2021-03-25 19:45:00+08:00                  2.0                153.0   130.0   106.0    88.5         44.2500
18      IG01           trace.self.row_count               31               2021-03-25 19:38:00+08:00                 16.0               1168.0  1028.0   816.0   705.0         44.0625
19  Tomcat01        trace.to_MG02.row_count               31               2021-03-25 19:32:00+08:00                  2.0                149.0   134.5   103.0    86.5         43.2500)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Validate required variables exist
for v in ('df_trace','thresh_trace','anomalies_trace'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# Incident window
start_ts = 1616671800
end_ts   = 1616673600

# Prepare windowed trace rows once
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
df_trace_window['value'] = pd.to_numeric(df_trace_window['value'], errors='coerce')

# Select top 10 anomalous trace groups by severity
top10_groups = (
    anomalies_trace
    .sort_values('severity_score', ascending=False)
    .head(10)[['cmdb_id','trace_name','p95','p5','count_in_window','severity_score']]
    .reset_index(drop=True)
)

# Helper safe division
def safe_div(numer, denom):
    if pd.isna(numer) or pd.isna(denom):
        return float('nan')
    denom = float(denom)
    if denom == 0:
        return SENTINEL
    return float(numer) / denom

# Collect all fault segments for selected groups
segments = []
for row in top10_groups.itertuples(index=False):
    cmdb = row.cmdb_id
    tname = row.trace_name
    p95 = float(row.p95) if pd.notna(row.p95) else float('nan')
    p5  = float(row.p5)  if pd.notna(row.p5)  else float('nan')
    count_in_window = int(row.count_in_window) if pd.notna(row.count_in_window) else 0

    # Filter rows for this group in window
    grp_rows = df_trace_window[(df_trace_window['cmdb_id']==cmdb) & (df_trace_window['trace_name']==tname)].copy()
    if grp_rows.empty:
        continue
    grp_rows = grp_rows.sort_values('timestamp').reset_index(drop=True)

    # Mark anomalous rows
    grp_rows['is_anom'] = (grp_rows['value'] > p95) | (grp_rows['value'] < p5)

    # Keep only anomalous rows
    anom_rows = grp_rows[grp_rows['is_anom']].copy()
    if anom_rows.empty:
        continue

    # Identify consecutive-minute segments: break where diff != 60
    anom_rows = anom_rows.sort_values('timestamp').reset_index(drop=True)
    anom_rows['ts_diff'] = anom_rows['timestamp'].diff().fillna(0).astype(int)
    # New segment flag
    anom_rows['new_seg'] = (anom_rows['ts_diff'] != 60).astype(int)
    anom_rows['seg_id'] = anom_rows['new_seg'].cumsum() - 1  # start from 0

    # Aggregate per segment
    seg_agg = (
        anom_rows
        .groupby('seg_id', dropna=False)
        .agg(
            cmdb_id = ('cmdb_id','first'),
            trace_name = ('trace_name','first'),
            segment_start_ts = ('timestamp','min'),
            segment_end_ts = ('timestamp','max'),
            segment_length_points = ('timestamp','count'),
            segment_min_value = ('value','min'),
            segment_max_value = ('value','max')
        )
        .reset_index(drop=True)
    )
    # Compute segment_length_minutes as inclusive minutes = points (since per-minute sampling)
    seg_agg['segment_length_minutes'] = seg_agg['segment_length_points']  # equivalently ((end-start)/60)+1

    # Compute segment_max_deviation
    def compute_dev(row):
        up = safe_div(row['segment_max_value'], p95)
        down = safe_div(p5, row['segment_min_value'])
        # handle nan: take max ignoring nan
        vals = [v for v in (up, down) if (not pd.isna(v))]
        return max(vals) if vals else float('nan')

    seg_agg['segment_max_deviation'] = seg_agg.apply(compute_dev, axis=1)

    # Add count_in_window and thresholds for context
    seg_agg['count_in_window'] = count_in_window
    seg_agg['p95'] = p95
    seg_agg['p5']  = p5
    seg_agg['severity_score_group'] = row.severity_score

    # Append
    segments.append(seg_agg)

# Concatenate all segments into full table trace_faults
if segments:
    trace_faults = pd.concat(segments, ignore_index=True)
else:
    trace_faults = pd.DataFrame(columns=[
        'cmdb_id','trace_name','segment_start_ts','segment_end_ts','segment_length_points',
        'segment_min_value','segment_max_value','segment_length_minutes','segment_max_deviation',
        'count_in_window','p95','p5','severity_score_group'
    ])

# For each (cmdb_id, trace_name) select earliest fault segment (smallest segment_start_ts)
if not trace_faults.empty:
    earliest_per_group = (
        trace_faults.sort_values('segment_start_ts', ascending=True)
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .first()
    )
else:
    earliest_per_group = trace_faults.copy()

# Convert epoch timestamps to Asia/Shanghai datetimes for display
def epoch_to_as_dt(epoch):
    if pd.isna(epoch):
        return pd.NaT
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz)

if not earliest_per_group.empty:
    earliest_per_group['earliest_fault_start_datetime_asia_shanghai'] = earliest_per_group['segment_start_ts'].apply(epoch_to_as_dt)
    earliest_per_group['earliest_fault_end_datetime_asia_shanghai'] = earliest_per_group['segment_end_ts'].apply(epoch_to_as_dt)

# Also mark the single overall earliest fault across all groups
if not earliest_per_group.empty:
    overall_idx = earliest_per_group['segment_start_ts'].idxmin()
    earliest_per_group['is_overall_earliest'] = False
    earliest_per_group.loc[overall_idx, 'is_overall_earliest'] = True
else:
    earliest_per_group['is_overall_earliest'] = pd.Series(dtype=bool)

# Prepare final compact table with requested columns, sorted by earliest_fault_start_timestamp ascending, limit 20
final_cols = [
    'cmdb_id','trace_name','count_in_window',
    'earliest_fault_start_datetime_asia_shanghai','earliest_fault_end_datetime_asia_shanghai',
    'segment_length_minutes','segment_min_value','segment_max_value','p95','p5','segment_max_deviation','is_overall_earliest'
]

if not earliest_per_group.empty:
    compact_table = (
        earliest_per_group[final_cols]
        .sort_values('earliest_fault_start_datetime_asia_shanghai', ascending=True)
        .reset_index(drop=True)
        .head(20)
        .copy()
    )
    # Round numeric columns for compactness
    for c in ['segment_min_value','segment_max_value','p95','p5','segment_max_deviation']:
        if c in compact_table.columns:
            compact_table[c] = pd.to_numeric(compact_table[c], errors='coerce').round(6)
else:
    compact_table = earliest_per_group[final_cols].head(20)

# Final results: trace_faults (full per-segment table) is stored in kernel variable,
# compact_table is the compact CSV-like result to return.
trace_faults, compact_table

```
Out[5]:
```
Summary of trace-level fault identification (window 2021-03-25 19:30:00+08 to 2021-03-25 20:00:00+08):

- Procedure: top 10 anomalous trace series (by severity) were examined in the incident window. For each series, per-minute rows were flagged when value > global P95 or < global P5, anomalous rows were grouped into consecutive-minute segments, and the earliest fault segment per series was returned. The full per-segment table is stored in kernel variable trace_faults.

- Overall earliest fault (across all inspected groups): 
  - IG02 — trace.self.row_count at 2021-03-25 19:32:00+08 (marked as the overall earliest).

- Earliest fault segment per selected group (compact list, earliest-first). All segments below are single-minute spikes (segment_length_minutes = 1):

  1) IG02 — trace.self.row_count  
     - earliest fault start/end: 2021-03-25 19:32:00+08  
     - count_in_window: 31, segment_min/max = 1026.0 / 1026.0  
     - global p95 = 1005.0, p5 = 646.0  
     - segment_max_deviation ≈ 1.0209  
     - overall earliest: True

  2) dockerB2 — trace.self.row_count  
     - earliest fault: 2021-03-25 19:36:00+08  
     - count_in_window: 31, segment_min/max = 1106.0 / 1106.0  
     - p95 = 1026.0, p5 = 396.5  
     - deviation ≈ 1.07797

  3) MG02 — trace.from_Tomcat02.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - count_in_window: 31, segment_min/max = 73.0 / 73.0  
     - p95 = 126.0, p5 = 74.0  
     - deviation ≈ 1.01370

  4) MG02 — trace.from_Tomcat03.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - segment_min/max = 68.0 / 68.0, p95=136.0, p5=75.0, deviation ≈ 1.10294

  5) Tomcat02 — trace.self.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - segment_min/max = 908.0 / 908.0, p95=1616.0, p5=987.0, deviation ≈ 1.08700

  6) Tomcat02 — trace.to_MG02.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - segment_min/max = 73.0 / 73.0, p95=126.0, p5=74.0, deviation ≈ 1.01370

  7) Tomcat03 — trace.self.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - segment_min/max = 1118.0 / 1118.0, p95=1807.0, p5=1140.0, deviation ≈ 1.01968

  8) Tomcat03 — trace.to_MG02.row_count  
     - earliest fault: 2021-03-25 19:37:00+08  
     - segment_min/max = 68.0 / 68.0, p95=136.0, p5=75.0, deviation ≈ 1.10294

  9) IG01 — trace.to_Tomcat02.row_count  
     - earliest fault: 2021-03-25 19:38:00+08  
     - segment_min/max = 72.0 / 72.0, p95=127.0, p5=73.0, deviation ≈ 1.01389

 10) Tomcat02 — trace.from_IG01.row_count  
     - earliest fault: 2021-03-25 19:38:00+08  
     - segment_min/max = 72.0 / 72.0, p95=127.0, p5=73.0, deviation ≈ 1.01389

- Observations and interpretation:
  - All reported earliest fault segments are isolated single-minute anomalies (no multi-minute consecutive segments among the top 10 groups).
  - Many anomalies are high or low spikes in row_count metrics (trace.self.row_count and inter-service row_count traces) affecting Tomcat/docker/IG/MG nodes.
  - Deviations are modest (most ≈ 1.01–1.10), indicating values slightly beyond thresholds; none (in this top set) triggered extremely large deviations except where sentinel handling would apply (not observed here for traces).
  - The single earliest detected event is IG02 trace.self.row_count at 19:32:00+08.

- Next steps:
  - Inspect service logs and request traffic around the listed timestamps for the implicated components (IG02, Tomcat02/03/04, MG02, dockerB2, IG01) to understand cause of spikes.
  - Correlate with metric and log anomalies (disk/CPU or request rate) at the same timestamps to identify root cause.
  - If repeated or prolonged segments are found in trace_faults (full table stored in kernel), escalate to service owners for deeper investigation.

If you want, I can display the compact CSV-like rows for these earliest segments or show more segments from the stored trace_faults variable.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                   trace_name  segment_start_ts  segment_end_ts  segment_length_points  segment_min_value  segment_max_value  segment_length_minutes  segment_max_deviation  count_in_window     p95     p5  severity_score_group
0   Tomcat02         trace.self.row_count        1616672220      1616672220                      1              908.0              908.0                       1               1.087004               31  1616.0  987.0               164.500
1   Tomcat02         trace.self.row_count        1616672880      1616672880                      1             1619.0             1619.0                       1               1.001856               31  1616.0  987.0               164.500
2   Tomcat02         trace.self.row_count        1616673240      1616673240                      1             1681.0             1681.0                       1               1.040223               31  1616.0  987.0               164.500
3   Tomcat02         trace.self.row_count        1616673600      1616673600                      1                6.0                6.0                       1             164.500000               31  1616.0  987.0               164.500
4   dockerB2         trace.self.row_count        1616672160      1616672160                      1             1106.0             1106.0                       1               1.077973               31  1026.0  396.5                99.125
..       ...                          ...               ...             ...                    ...                ...                ...                     ...                    ...              ...     ...    ...                   ...
32  Tomcat02    trace.from_IG01.row_count        1616673600      1616673600                      1                1.0                1.0                       1              73.000000               31   127.0   73.0                73.000
33      IG01  trace.to_Tomcat02.row_count        1616672280      1616672280                      1               72.0               72.0                       1               1.013889               31   127.0   73.0                73.000
34      IG01  trace.to_Tomcat02.row_count        1616673120      1616673120                      1              137.0              137.0                       1               1.078740               31   127.0   73.0                73.000
35      IG01  trace.to_Tomcat02.row_count        1616673240      1616673240                      1              131.0              131.0                       1               1.031496               31   127.0   73.0                73.000
36      IG01  trace.to_Tomcat02.row_count        1616673600      1616673600                      1                1.0                1.0                       1              73.000000               31   127.0   73.0                73.000

[37 rows x 13 columns],     cmdb_id                     trace_name  count_in_window earliest_fault_start_datetime_asia_shanghai earliest_fault_end_datetime_asia_shanghai  segment_length_minutes  segment_min_value  segment_max_value     p95      p5  segment_max_deviation  is_overall_earliest
0      IG02           trace.self.row_count               31                   2021-03-25 19:32:00+08:00                 2021-03-25 19:32:00+08:00                       1             1026.0             1026.0  1005.0   646.0               1.020896                 True
1  dockerB2           trace.self.row_count               31                   2021-03-25 19:36:00+08:00                 2021-03-25 19:36:00+08:00                       1             1106.0             1106.0  1026.0   396.5               1.077973                False
2      MG02  trace.from_Tomcat02.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1               73.0               73.0   126.0    74.0               1.013699                False
3      MG02  trace.from_Tomcat03.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1               68.0               68.0   136.0    75.0               1.102941                False
4  Tomcat02           trace.self.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1              908.0              908.0  1616.0   987.0               1.087004                False
5  Tomcat02        trace.to_MG02.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1               73.0               73.0   126.0    74.0               1.013699                False
6  Tomcat03           trace.self.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1             1118.0             1118.0  1807.0  1140.0               1.019678                False
7  Tomcat03        trace.to_MG02.row_count               31                   2021-03-25 19:37:00+08:00                 2021-03-25 19:37:00+08:00                       1               68.0               68.0   136.0    75.0               1.102941                False
8      IG01    trace.to_Tomcat02.row_count               31                   2021-03-25 19:38:00+08:00                 2021-03-25 19:38:00+08:00                       1               72.0               72.0   127.0    73.0               1.013889                False
9  Tomcat02      trace.from_IG01.row_count               31                   2021-03-25 19:38:00+08:00                 2021-03-25 19:38:00+08:00                       1               72.0               72.0   127.0    73.0               1.013889                False)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Validate presence
for v in ('anomalies_metric','df_metric','thresh_metric'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# Candidate components
candidates = ['IG02','IG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','apache01','apache02']

# Work on copy
am = anomalies_metric.copy()

# Ensure metric name column is 'kpi_name'
if 'kpi_name' not in am.columns and 'metric_name' in am.columns:
    am = am.rename(columns={'metric_name':'kpi_name'})

# Keep only candidate components
am_sel = am[am['cmdb_id'].isin(candidates)].copy()

# Ensure earliest_anomaly_ts exists (epoch seconds). If not, derive from datetime field.
if 'earliest_anomaly_ts' not in am_sel.columns:
    if 'earliest_anomaly_datetime_asia_shanghai' in am_sel.columns:
        def dt_to_epoch(d):
            if pd.isna(d):
                return pd.NA
            # Ensure it's a timezone-aware Timestamp
            ts = pd.to_datetime(d)
            if ts.tzinfo is None:
                ts = ts.tz_localize(tz)
            return int(ts.tz_convert('UTC').timestamp())
        am_sel['earliest_anomaly_ts'] = am_sel['earliest_anomaly_datetime_asia_shanghai'].apply(dt_to_epoch)
    else:
        am_sel['earliest_anomaly_ts'] = pd.NA

# Drop rows lacking earliest anomaly epoch
am_sel = am_sel[am_sel['earliest_anomaly_ts'].notna()].copy()
if am_sel.empty:
    selected_metric_anoms = pd.DataFrame()  # nothing to return
else:
    # Ensure integer epoch
    am_sel['earliest_anomaly_ts'] = am_sel['earliest_anomaly_ts'].astype(int)

    # Prepare earliest_anomaly_datetime_asia_shanghai column (in case missing)
    def epoch_to_asia_shanghai(epoch):
        return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz)
    am_sel['earliest_anomaly_datetime_asia_shanghai'] = am_sel['earliest_anomaly_ts'].apply(epoch_to_asia_shanghai)

    # Merge with df_metric to get the sample value at the earliest anomaly timestamp
    # Left keys: cmdb_id, kpi_name, earliest_anomaly_ts  -> Right keys: cmdb_id, kpi_name, timestamp
    # Ensure df_metric value numeric
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

    merged = am_sel.merge(
        df_metric[['cmdb_id','kpi_name','timestamp','value']].rename(columns={'timestamp':'earliest_anomaly_ts', 'value':'earliest_sample_value'}),
        on=['cmdb_id','kpi_name','earliest_anomaly_ts'],
        how='left'
    )

    # If merge found no match (NaN earliest_sample_value), keep NA
    # Select requested columns, ensuring they exist
    out_cols = [
        'cmdb_id','kpi_name','count_in_window','earliest_anomaly_ts','earliest_anomaly_datetime_asia_shanghai',
        'min_value_in_window','max_value_in_window','p95','p50','p5','severity_score','earliest_sample_value'
    ]
    available = [c for c in out_cols if c in merged.columns]
    selected_metric_anoms = merged[available].copy()

    # Round numeric columns for compactness
    for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score','earliest_sample_value']:
        if c in selected_metric_anoms.columns:
            selected_metric_anoms[c] = pd.to_numeric(selected_metric_anoms[c], errors='coerce').round(6)

    # Sort by earliest_anomaly_ts ascending and limit to top 20
    selected_metric_anoms = selected_metric_anoms.sort_values('earliest_anomaly_ts', ascending=True).head(20).reset_index(drop=True)

# Store in kernel variable as requested
selected_metric_anoms

```
Out[6]:
```
Summary of selected anomalous metric groups for the specified candidate components:

- Scope: 20 anomalous (cmdb_id, kpi_name) groups for components IG02, IG01, Tomcat01–04, MG01–02, apache01–02 were selected from anomalies_metric. All reported earliest anomaly timestamps are at or after 1616671800 (2021-03-25 19:30:00+08). The full result is stored in kernel variable selected_metric_anoms.

Key findings (top examples from the returned table):
- IG01 — JVM-Memory_HeapMemoryUsage
  - earliest anomaly: 1616671800 (2021-03-25 19:30:00+08)
  - count_in_window: 20, min=3.4171, max=41.4453
  - global p95=39.8189, p50=22.7176, p5=5.55346
  - severity ≈ 1.625; earliest sample value = 3.4171

- MG01 — Several OS metrics (CPU SysTime, CPU idle, filesystem, memory cache)
  - many earliest anomalies at 1616671800 (19:30+08)
  - examples: OSLinux-CPU_CPU_CPUSysTime (count 30, min 0.4947, max 0.8067, p95≈0.7944, severity≈1.0498), OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable (count 11, p95≈21096.5, severity≈1.00026)
  - earliest_sample_values are provided per row.

- Tomcat02 — Request-related metrics
  - Tomcat-Requests…RequestCount and Processing Time show earliest_anomaly at 1616671800 with large numeric values (earliest_sample_value ~3.5037e6 and ~2.1814e9 respectively) and small severity >1 (≈1.0001 and 1.00004), indicating values slightly beyond global P95.

- Tomcat03 — multiple filesystem and disk metrics (some abnormal)
  - Several disk metrics have min_value_in_window == 0 (e.g., DSKBps, DSKPer...), producing a sentinel severity (1e18) because p5/min_value triggers division-by-zero handling.
  - Example: OSLinux-LOCALDISK-sda_DSKBps: min=0, max=5, p95=2.9, p5=0 → sentinel severity = 1e18; earliest_sample_value = 5.0

Notable patterns and interpretation:
- Many earliest anomalies occur at the start of the window (2021-03-25 19:30:00+08).
- Several filesystem/local-disk metrics show zeros (min_value_in_window == 0). These zeros cause the computed severity to be set to a large sentinel (1e18) due to division-by-zero safe handling — this often indicates either true zero I/O samples or missing/garbled agent readings and should be investigated.
- Most CPU, memory, and request-count anomalies have modest severity (>1 but close to 1), meaning values slightly beyond the P95; Tomcat request metrics show large absolute values but small relative deviations.
- earliest_sample_value column gives the actual metric value at the earliest anomalous timestamp for quick inspection.

Recommended next steps:
- Investigate disk-agent/collector behavior on nodes with sentinel severities (Tomcat03 and others) to determine if zeros are valid or due to collection issues.
- For Tomcat request and processing metrics, correlate with trace-level anomalies and application logs around 19:30 to 19:40 to see if traffic spikes or processing delays occurred.
- For IG01 JVM memory anomaly (low sample at earliest timestamp), check JVM logs and GC events near 19:30 to rule out transient memory pressure.

If you want, I can display the stored selected_metric_anoms table as CSV-like rows or filter it further by severity or component.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  earliest_anomaly_ts earliest_anomaly_datetime_asia_shanghai  min_value_in_window  max_value_in_window           p95           p50            p5  severity_score  earliest_sample_value
0       IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage               20           1616671800               2021-03-25 19:30:00+08:00         3.417100e+00         4.144530e+01  3.981890e+01  2.271760e+01  5.553460e+00    1.625197e+00           3.417100e+00
1       MG01                         OSLinux-CPU_CPU_CPUSysTime               30           1616671800               2021-03-25 19:30:00+08:00         4.947000e-01         8.067200e-01  7.943560e-01  6.239470e-01  5.193310e-01    1.049790e+00           8.067200e-01
2       MG01                        OSLinux-CPU_CPU_CPUidleutil               30           1616671800               2021-03-25 19:30:00+08:00         7.227212e+01         7.351130e+01  7.344107e+01  7.317031e+01  7.260368e+01    1.004588e+00           7.227212e+01
3       MG01                    OSLinux-CPU_CPU-0_SingleCpuidle               15           1616671800               2021-03-25 19:30:00+08:00         9.670600e+01         9.805390e+01  9.796101e+01  9.737410e+01  9.679956e+01    1.000967e+00           9.670600e+01
4   Tomcat03      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity               10           1616671800               2021-03-25 19:30:00+08:00         5.563470e+01         5.579750e+01  5.578724e+01  5.570655e+01  5.563992e+01    1.000184e+00           5.563470e+01
5   Tomcat03  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...               11           1616671800               2021-03-25 19:30:00+08:00         2.102400e+04         2.110200e+04  2.109650e+04  2.105800e+04  2.102950e+04    1.000262e+00           2.110200e+04
6       MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem               19           1616671800               2021-03-25 19:30:00+08:00         5.465000e+03         5.492000e+03  5.490200e+03  5.485000e+03  5.468600e+03    1.000659e+00           5.465000e+03
7   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...               31           1616671800               2021-03-25 19:30:00+08:00         3.503695e+06         3.509916e+06  3.509578e+06  3.506676e+06  3.504044e+06    1.000100e+00           3.503695e+06
8   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...               31           1616671800               2021-03-25 19:30:00+08:00         2.181429e+09         2.182804e+09  2.182718e+09  2.182080e+09  2.181520e+09    1.000042e+00           2.181429e+09
9       MG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...                9           1616671800               2021-03-25 19:30:00+08:00         2.232000e-01         2.260000e-01  2.258400e-01  2.245000e-01  2.233600e-01    1.000717e+00           2.232000e-01
10      MG01      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity                9           1616671800               2021-03-25 19:30:00+08:00         7.257970e+01         7.355630e+01  7.350082e+01  7.306530e+01  7.264066e+01    1.000840e+00           7.257970e+01
11      MG01        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity                9           1616671800               2021-03-25 19:30:00+08:00         2.757500e+00         2.757800e+00  2.757800e+00  2.757700e+00  2.757540e+00    1.000015e+00           2.757500e+00
12  Tomcat03     OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace               11           1616671800               2021-03-25 19:30:00+08:00         2.646200e+04         2.653900e+04  2.653350e+04  2.650600e+04  2.646750e+04    1.000208e+00           2.646200e+04
13  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               15           1616671800               2021-03-25 19:30:00+08:00         0.000000e+00         5.000000e+00  2.900000e+00  0.000000e+00  0.000000e+00    1.000000e+18           5.000000e+00
14      IG01                    OSLinux-CPU_CPU-2_SingleCpuidle               10           1616671800               2021-03-25 19:30:00+08:00         9.858590e+01         9.916820e+01  9.910084e+01  9.885200e+01  9.862343e+01    1.000680e+00           9.858590e+01
15  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...               15           1616671800               2021-03-25 19:30:00+08:00         0.000000e+00         1.200000e+00  5.700000e-01  0.000000e+00  0.000000e+00    1.000000e+18           1.200000e+00
16  Tomcat03                         OSLinux-CPU_CPU_CPUSysTime               30           1616671800               2021-03-25 19:30:00+08:00         3.494000e-01         6.154100e-01  6.000300e-01  5.177080e-01  3.609150e-01    1.032956e+00           6.111000e-01
17  Tomcat03                        OSLinux-CPU_CPU_CPUUserTime               30           1616671800               2021-03-25 19:30:00+08:00         2.537020e+01         2.575290e+01  2.564495e+01  2.551311e+01  2.538631e+01    1.004209e+00           2.564955e+01
18  Tomcat03                        OSLinux-CPU_CPU_CPUidleutil               30           1616671800               2021-03-25 19:30:00+08:00         7.192297e+01         7.396860e+01  7.392987e+01  7.355181e+01  7.300406e+01    1.015031e+00           7.298275e+01
19  Tomcat03            OSLinux-OSLinux_FILESYSTEM_-_FSCapacity               10           1616671800               2021-03-25 19:30:00+08:00         3.083980e+01         3.083990e+01  3.083990e+01  3.083990e+01  3.083985e+01    1.000001e+00           3.083980e+01```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Reuse existing dataframes from kernel: df_log, thresh_log, df_err
# Validate presence
for v in ('df_log','thresh_log','df_err'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# Candidate components and incident window
candidates = ['IG02','IG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','apache01','apache02']
start_ts = 1616671800
end_ts   = 1616673600

# Windowed logs for candidate components
df_log_window = df_log[
    (df_log['cmdb_id'].isin(candidates)) &
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts)
].copy()
df_log_window['value'] = pd.to_numeric(df_log_window['value'], errors='coerce')

# Prepare global thresholds: thresh_log has 'metric_name' for the log_name
# Ensure numeric
thresh_log['p95'] = pd.to_numeric(thresh_log['p95'], errors='coerce')
thresh_log['p50'] = pd.to_numeric(thresh_log['p50'], errors='coerce')
thresh_log['p5']  = pd.to_numeric(thresh_log['p5'], errors='coerce')

# Per-group summary within window
group_win = (
    df_log_window
    .groupby(['cmdb_id','log_name'], dropna=False)
    .agg(
        count_in_window = ('value','count'),
        earliest_ts = ('timestamp','min'),
        min_value_in_window = ('value','min'),
        max_value_in_window = ('value','max')
    )
    .reset_index()
)

# Merge global thresholds from thresh_log (metric_name -> log_name)
group_win = group_win.merge(
    thresh_log.rename(columns={'metric_name':'log_name'}),
    on=['cmdb_id','log_name'],
    how='left'
)

# Flag anomalous groups where max>p95 or min<p5
group_win['anomaly'] = (
    (group_win['max_value_in_window'] > group_win['p95']) |
    (group_win['min_value_in_window'] < group_win['p5'])
)

# Row-level anomalies to find earliest anomaly timestamp per group
rows_with_thresh = df_log_window.merge(
    thresh_log.rename(columns={'metric_name':'log_name'}),
    on=['cmdb_id','log_name'],
    how='left'
)
rows_with_thresh['is_anom_row'] = (rows_with_thresh['value'] > rows_with_thresh['p95']) | (rows_with_thresh['value'] < rows_with_thresh['p5'])

earliest_anom = (
    rows_with_thresh[rows_with_thresh['is_anom_row']]
    .groupby(['cmdb_id','log_name'], dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_ts'})
)

# Join earliest anomaly ts into group_win
group_win = group_win.merge(earliest_anom, on=['cmdb_id','log_name'], how='left')

# Safe division helper
def safe_div(numer, denom):
    if pd.isna(numer) or pd.isna(denom):
        return float('nan')
    denom = float(denom)
    if denom == 0:
        return SENTINEL
    return float(numer) / denom

group_win['ratio_up'] = group_win.apply(lambda r: safe_div(r['max_value_in_window'], r['p95']), axis=1)
group_win['ratio_down'] = group_win.apply(lambda r: safe_div(r['p5'], r['min_value_in_window']), axis=1)
group_win['severity_score'] = group_win[['ratio_up','ratio_down']].max(axis=1)

# Filter only anomalous groups and prepare full result variable anomalies_log
anomalies_log = group_win[group_win['anomaly']].copy()

# Add datetime conversion for earliest_anomaly_ts and earliest_ts for display
def epoch_to_as_dt(e):
    if pd.isna(e):
        return pd.NaT
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz)

anomalies_log['earliest_anomaly_datetime_asia_shanghai'] = anomalies_log['earliest_anomaly_ts'].apply(lambda e: epoch_to_as_dt(e))
anomalies_log['earliest_window_datetime_asia_shanghai'] = anomalies_log['earliest_ts'].apply(lambda e: epoch_to_as_dt(e))

# Compact top-20 by earliest_anomaly_ts ascending
anomalies_log_top = (
    anomalies_log
    .sort_values('earliest_anomaly_ts', ascending=True)
    .reset_index(drop=True)
    [['cmdb_id','log_name','count_in_window','earliest_anomaly_ts','earliest_anomaly_datetime_asia_shanghai',
      'min_value_in_window','max_value_in_window','p95','p50','p5','severity_score']]
    .head(20)
    .copy()
)

# Round numeric columns for compactness
for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score']:
    if c in anomalies_log_top.columns:
        anomalies_log_top[c] = pd.to_numeric(anomalies_log_top[c], errors='coerce').round(6)

# 2) Scan error_logs.csv for candidate components within window; return up to 10 sample rows
df_err_window = df_err[
    (df_err['cmdb_id'].isin(candidates)) &
    (df_err['timestamp'] >= start_ts) &
    (df_err['timestamp'] <= end_ts)
].copy()

if df_err_window.empty:
    err_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    err_empty = True
else:
    err_samples = df_err_window.sort_values('timestamp').loc[:, ['timestamp','cmdb_id','message']].head(10).copy()
    # add datetime for convenience
    err_samples['as_datetime_asia_shanghai'] = pd.to_datetime(err_samples['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    err_empty = False

# Final displayed variables (compact outputs)
anomalies_log_top, err_samples, err_empty

```
Out[7]:
```
Summary of log and error-log scans for candidate components within window 1616671800–1616673600 (2021-03-25 19:30–20:00 +08):

1) Log anomalies (top groups found; shown earliest-first)
- Tomcat01 — log.row_count  
  • earliest anomaly: 2021-03-25 19:32:00+08 (epoch 1616671920)  
  • count_in_window=31, min=5, max=291, p95=265.0, p50=204.0, p5=174.5  
  • severity_score ≈ 34.90 (max >> p95)

- apache01 — log.row_count  
  • earliest anomaly: 2021-03-25 19:32:00+08  
  • count=31, min=7, max=547, p95=495.5, p50=405.0, p5=340.5  
  • severity_score ≈ 48.64

- Tomcat03 — log.row_count  
  • earliest anomaly: 2021-03-25 19:37:00+08  
  • count=31, min=3, max=283, p95=268.0, p50=210.0, p5=177.0  
  • severity_score ≈ 59.00

- Tomcat02 — log.row_count  
  • earliest anomaly: 2021-03-25 19:37:00+08  
  • count=31, min=1, max=250, p95=243.0, p50=206.0, p5=145.5  
  • severity_score ≈ 145.50 (very large relative deviation due to tiny min_value)

- apache02 — log.row_count  
  • earliest anomaly: 2021-03-25 19:37:00+08  
  • count=31, min=5, max=522, p95=511.5, p50=418.0, p5=341.5  
  • severity_score ≈ 68.30

- Tomcat03 — log.gc_pause_max  
  • earliest anomaly: 2021-03-25 19:39:00+08  
  • count=10, min=0.03, max=0.08, p95=0.08, p50=0.07, p5=0.0435  
  • severity_score ≈ 1.45

- Tomcat01 — log.gc_pause_max  
  • earliest anomaly: 2021-03-25 19:43:00+08  
  • count=10, min=0.02, max=0.09, p95=0.0855, p50=0.065, p5=0.0245  
  • severity_score ≈ 1.225

- Tomcat02 — log.gc_pause_max  
  • earliest anomaly: 2021-03-25 19:44:00+08  
  • count=10, min=0.03, max=0.10, p95=0.0910, p50=0.07, p5=0.03  
  • severity_score ≈ 1.099

- Tomcat04 — log.row_count  
  • earliest anomaly: 2021-03-25 19:46:00+08  
  • count=31, min=5, max=262, p95=258.0, p50=217.0, p5=163.0  
  • severity_score ≈ 32.60

- Tomcat04 — log.gc_pause_max  
  • earliest anomaly: 2021-03-25 19:58:00+08  
  • count=12, min=0.03, max=0.11, p95=0.11, p50=0.07, p5=0.0355  
  • severity_score ≈ 1.183

(These are the top anomalous log groups; full anomalous-log table is stored in anomalies_log.)

Interpretation of log anomalies:
- Several web/server access metrics (log.row_count) show large spikes relative to their p95 (high severity), especially for Tomcat and apache instances around 19:32–19:37. This suggests sudden traffic/row-count surges.
- GC pause anomalies (log.gc_pause_max) appear later (19:39–19:58) on Tomcat nodes with modest relative deviations (>1), indicating some GC events worth checking.
- Some very large severity values reflect tiny min values or very large max values relative to p95/p5 and may warrant checking raw samples and ingestion.

2) error_logs.csv samples for candidate components in window
- No error log rows were found for these candidate components in the incident window. Result: empty (no samples).

Recommended next steps:
- Correlate the log.row_count spikes at ~19:32–19:37 with trace and metric anomalies (request counts, CPU, disk) to determine whether traffic spikes caused downstream effects.
- Inspect Tomcat GC pause entries around 19:39–19:58 if application latency or throughput issues were observed.
- Since error_logs is empty, rely on application logs and traces for root-cause details; consider collecting more verbose logs if needed.

If you want, I can show the compact CSV-like rows for anomalies_log or export the full anomalies_log table stored in kernel.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  count_in_window  earliest_anomaly_ts earliest_anomaly_datetime_asia_shanghai  min_value_in_window  max_value_in_window       p95      p50        p5  severity_score
0  Tomcat01     log.row_count               31           1616671920               2021-03-25 19:32:00+08:00                 5.00               291.00  265.0000  204.000  174.5000       34.900000
1  apache01     log.row_count               31           1616671920               2021-03-25 19:32:00+08:00                 7.00               547.00  495.5000  405.000  340.5000       48.642857
2  Tomcat03     log.row_count               31           1616672220               2021-03-25 19:37:00+08:00                 3.00               283.00  268.0000  210.000  177.0000       59.000000
3  Tomcat02     log.row_count               31           1616672220               2021-03-25 19:37:00+08:00                 1.00               250.00  243.0000  206.000  145.5000      145.500000
4  apache02     log.row_count               31           1616672220               2021-03-25 19:37:00+08:00                 5.00               522.00  511.5000  418.000  341.5000       68.300000
5  Tomcat03  log.gc_pause_max               10           1616672340               2021-03-25 19:39:00+08:00                 0.03                 0.08    0.0800    0.070    0.0435        1.450000
6  Tomcat01  log.gc_pause_max               10           1616672580               2021-03-25 19:43:00+08:00                 0.02                 0.09    0.0855    0.065    0.0245        1.225000
7  Tomcat02  log.gc_pause_max               10           1616672640               2021-03-25 19:44:00+08:00                 0.03                 0.10    0.0910    0.070    0.0300        1.098901
8  Tomcat04     log.row_count               31           1616672760               2021-03-25 19:46:00+08:00                 5.00               262.00  258.0000  217.000  163.0000       32.600000
9  Tomcat04  log.gc_pause_max               12           1616673480               2021-03-25 19:58:00+08:00                 0.03                 0.11    0.1100    0.070    0.0355        1.183333, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], True)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Candidate components
components = ['IG02','IG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','apache01','apache02']

# Ensure required kernel variables exist
for v in ('selected_metric_anoms','anomalies_trace','anomalies_log'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# Reuse selected_metric_anoms (metric anomalies already filtered to candidates earlier)
metrics = selected_metric_anoms.copy()

# Prepare metric per-component aggregation
if not metrics.empty:
    metrics_group = (
        metrics
        .groupby('cmdb_id', dropna=False)
        .agg(
            earliest_metric_anomaly_timestamp = ('earliest_anomaly_ts', 'min'),
            earliest_metric_severity = ('severity_score', 'max'),
            number_of_metric_anomalous_groups = ('kpi_name','nunique')
        )
        .reset_index()
    )
else:
    metrics_group = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_timestamp','earliest_metric_severity','number_of_metric_anomalous_groups'])

# Add datetime column
def epoch_to_asia_shanghai(e):
    if pd.isna(e):
        return pd.NaT
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz)

metrics_group['earliest_metric_anomaly_datetime_asia_shanghai'] = metrics_group['earliest_metric_anomaly_timestamp'].apply(lambda e: epoch_to_asia_shanghai(e))

# Prepare trace per-component aggregation from anomalies_trace
tr = anomalies_trace.copy() if 'anomalies_trace' in globals() else pd.DataFrame()
if not tr.empty:
    # Ensure earliest_anomaly_ts exists in anomalies_trace
    if 'earliest_anomaly_ts' not in tr.columns:
        # try to use datetime column if present
        if 'earliest_anomaly_datetime_asia_shanghai' in tr.columns:
            tr['earliest_anomaly_ts'] = tr['earliest_anomaly_datetime_asia_shanghai'].apply(
                lambda d: int(d.tz_convert('UTC').timestamp()) if pd.notna(d) else pd.NA
            )
    trace_group = (
        tr.groupby('cmdb_id', dropna=False)
        .agg(
            earliest_trace_anomaly_timestamp = ('earliest_anomaly_ts','min'),
            earliest_trace_severity = ('severity_score','max'),
            number_of_trace_anomalous_groups = ('trace_name','nunique')
        )
        .reset_index()
    )
else:
    trace_group = pd.DataFrame(columns=['cmdb_id','earliest_trace_anomaly_timestamp','earliest_trace_severity','number_of_trace_anomalous_groups'])

trace_group['earliest_trace_anomaly_datetime_asia_shanghai'] = trace_group['earliest_trace_anomaly_timestamp'].apply(lambda e: epoch_to_asia_shanghai(e))

# Prepare log per-component aggregation from anomalies_log
lg = anomalies_log.copy() if 'anomalies_log' in globals() else pd.DataFrame()
if not lg.empty:
    # anomalies_log already has earliest_anomaly_ts
    log_group = (
        lg.groupby('cmdb_id', dropna=False)
        .agg(
            earliest_log_anomaly_timestamp = ('earliest_anomaly_ts','min'),
            earliest_log_severity = ('severity_score','max'),
            number_of_log_anomalous_groups = ('log_name','nunique')
        )
        .reset_index()
    )
else:
    log_group = pd.DataFrame(columns=['cmdb_id','earliest_log_anomaly_timestamp','earliest_log_severity','number_of_log_anomalous_groups'])

log_group['earliest_log_anomaly_datetime_asia_shanghai'] = log_group['earliest_log_anomaly_timestamp'].apply(lambda e: epoch_to_asia_shanghai(e))

# Build base DataFrame for candidate components
corr = pd.DataFrame({'cmdb_id': components})

# Merge metric, trace, log summaries
corr = corr.merge(metrics_group, on='cmdb_id', how='left')
corr = corr.merge(trace_group, on='cmdb_id', how='left')
corr = corr.merge(log_group, on='cmdb_id', how='left')

# Replace missing numeric fields with NA (keep NaN for pandas)
# Compute earliest non-null anomaly timestamp across metric/trace/log (epoch)
corr['earliest_any_anomaly_ts'] = corr[
    ['earliest_metric_anomaly_timestamp','earliest_trace_anomaly_timestamp','earliest_log_anomaly_timestamp']
].min(axis=1, skipna=True)

# Convert earliest_any_anomaly_ts to datetime (Asia/Shanghai)
corr['earliest_any_anomaly_datetime_asia_shanghai'] = corr['earliest_any_anomaly_ts'].apply(lambda e: epoch_to_asia_shanghai(e))

# Round severity fields for compactness
for sev in ['earliest_metric_severity','earliest_trace_severity','earliest_log_severity']:
    if sev in corr.columns:
        corr[sev] = pd.to_numeric(corr[sev], errors='coerce').round(6)

# Reorder columns as requested
cols = [
    'cmdb_id',
    'earliest_metric_anomaly_timestamp','earliest_metric_anomaly_datetime_asia_shanghai','earliest_metric_severity','number_of_metric_anomalous_groups',
    'earliest_trace_anomaly_timestamp','earliest_trace_anomaly_datetime_asia_shanghai','earliest_trace_severity','number_of_trace_anomalous_groups',
    'earliest_log_anomaly_timestamp','earliest_log_anomaly_datetime_asia_shanghai','earliest_log_severity','number_of_log_anomalous_groups',
    'earliest_any_anomaly_ts','earliest_any_anomaly_datetime_asia_shanghai'
]
# Some columns may not exist; filter available
cols = [c for c in cols if c in corr.columns]
correlation_summary = corr[cols].copy()

# Sort by earliest_any_anomaly_datetime_asia_shanghai ascending, with NaT last
correlation_summary = correlation_summary.sort_values('earliest_any_anomaly_ts', ascending=True, na_position='last').reset_index(drop=True)

# Limit to top 20 (we have 10 components)
correlation_summary = correlation_summary.head(20)

# Ensure compactness: display selected columns only
correlation_summary

```
Out[8]:
```
Summary correlation for the candidate components (window 2021-03-25 19:30–20:00 +08):

General observation
- Most components show their earliest detected anomaly at or shortly after 19:30+08. The table of components was sorted by the earliest non-null anomaly time; many components have anomalies across multiple telemetry types (metric, trace, log).

Per-component highlights (earliest anomaly time and which telemetry types had anomalies)
- IG02
  - Earliest anomaly: 2021-03-25 19:30:00+08
  - Trace: earliest at 19:30 (earliest_trace_severity ≈ 80.75, number_of_trace_anomalous_groups = 15)
  - No metric or log anomalies recorded (metric/log fields are NA)

- IG01
  - Earliest anomaly: 2021-03-25 19:30:00+08 (metric) and 19:31:00+08 (trace)
  - Metric: earliest_metric at 19:30, earliest_metric_severity ≈ 1.625, number_of_metric_anomalous_groups = 2
  - Trace: earliest_trace at 19:31, earliest_trace_severity ≈ 73.00, number_of_trace_anomalous_groups = 15
  - No log anomalies recorded

- Tomcat01
  - Earliest anomaly: 2021-03-25 19:30:00+08 (trace)
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 43.25, number_of_trace_anomalous_groups = 18
  - Log: earliest_log at 19:32:00+08 (log.row_count), earliest_log_severity ≈ 34.90, number_of_log_anomalous_groups = 2
  - No metric anomalies recorded

- Tomcat02
  - Earliest anomaly: 2021-03-25 19:30:00+08 (metric & trace)
  - Metric: earliest_metric at 19:30, earliest_metric_severity ≈ 1.0001, number_of_metric_anomalous_groups = 2
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 164.50 (high), number_of_trace_anomalous_groups = 15
  - Log: earliest_log at 19:37:00+08 (log.row_count), earliest_log_severity ≈ 145.5, number_of_log_anomalous_groups = 2

- Tomcat03
  - Earliest anomaly: 2021-03-25 19:30:00+08 (metric & trace)
  - Metric: earliest_metric shows sentinel severity (1e18) for some disk metrics due to zero min values (number_of_metric_anomalous_groups = 9)
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 76.00, number_of_trace_anomalous_groups = 15
  - Log: earliest_log at 19:37:00+08 (log.row_count), earliest_log_severity ≈ 59.00, number_of_log_anomalous_groups = 2

- Tomcat04
  - Earliest anomaly: 2021-03-25 19:30:00+08 (trace)
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 44.25, number_of_trace_anomalous_groups = 15
  - Log: earliest_log at 19:46:00+08 (log.row_count), earliest_log_severity ≈ 32.60, number_of_log_anomalous_groups = 2
  - No metric anomalies recorded

- MG01
  - Earliest anomaly: 2021-03-25 19:30:00+08 (metric & trace)
  - Metric: earliest_metric at 19:30, earliest_metric_severity ≈ 1.0498, number_of_metric_anomalous_groups = 7
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 53.50, number_of_trace_anomalous_groups = 39
  - No log anomalies recorded

- MG02
  - Earliest anomaly: 2021-03-25 19:30:00+08 (trace)
  - Trace: earliest_trace at 19:30, earliest_trace_severity ≈ 75.00, number_of_trace_anomalous_groups = 39
  - No metric or log anomalies recorded

- apache01
  - Earliest anomaly: 2021-03-25 19:32:00+08 (log)
  - Log: earliest_log at 19:32, earliest_log_severity ≈ 48.64, number_of_log_anomalous_groups = 1
  - No metric or trace anomalies recorded

- apache02
  - Earliest anomaly: 2021-03-25 19:37:00+08 (log)
  - Log: earliest_log at 19:37, earliest_log_severity ≈ 68.30, number_of_log_anomalous_groups = 1
  - No metric or trace anomalies recorded

Key takeaways and recommended next steps
- Many components have trace anomalies with high severities (IG02, MG02, Tomcat02, Tomcat03), indicating traffic/row-count spikes or inter-service count anomalies concentrated at ~19:30–19:37.
- Log spikes (log.row_count) on Tomcat and Apache nodes cluster around 19:32–19:37 — correlate these with the trace spikes to see if traffic surges caused downstream effects.
- Several metric disk/local-disk series contain zero min values producing sentinel severities (e.g., Tomcat03). Investigate whether these zeros are true zeros or collection/agent issues.
- Prioritize investigation on components with both trace and log spikes (Tomcat02, Tomcat03, Tomcat01) and on IG/MG nodes showing high trace severities (IG02, MG02).

The merged table has been stored in kernel variable correlation_summary for further inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  earliest_metric_anomaly_timestamp earliest_metric_anomaly_datetime_asia_shanghai  earliest_metric_severity  number_of_metric_anomalous_groups  earliest_trace_anomaly_timestamp earliest_trace_anomaly_datetime_asia_shanghai  earliest_trace_severity  number_of_trace_anomalous_groups  earliest_log_anomaly_timestamp earliest_log_anomaly_datetime_asia_shanghai  earliest_log_severity  number_of_log_anomalous_groups  \
0      IG02                                NaN                                            NaT                       NaN                                NaN                        1616671800                     2021-03-25 19:30:00+08:00                    80.75                              15.0                            <NA>                                         NaT                    NaN                             NaN   
1      IG01                       1.616672e+09                      2021-03-25 19:30:00+08:00              1.625197e+00                                2.0                        1616671860                     2021-03-25 19:31:00+08:00                    73.00                              15.0                            <NA>                                         NaT                    NaN                             NaN   
2  Tomcat01                                NaN                                            NaT                       NaN                                NaN                        1616671800                     2021-03-25 19:30:00+08:00                    43.25                              18.0                      1616671920                   2021-03-25 19:32:00+08:00              34.900000                             2.0   
3  Tomcat02                       1.616672e+09                      2021-03-25 19:30:00+08:00              1.000100e+00                                2.0                        1616671800                     2021-03-25 19:30:00+08:00                   164.50                              15.0                      1616672220                   2021-03-25 19:37:00+08:00             145.500000                             2.0   
4  Tomcat03                       1.616672e+09                      2021-03-25 19:30:00+08:00              1.000000e+18                                9.0                        1616671800                     2021-03-25 19:30:00+08:00                    76.00                              15.0                      1616672220                   2021-03-25 19:37:00+08:00              59.000000                             2.0   
5  Tomcat04                                NaN                                            NaT                       NaN                                NaN                        1616671800                     2021-03-25 19:30:00+08:00                    44.25                              15.0                      1616672760                   2021-03-25 19:46:00+08:00              32.600000                             2.0   
6      MG01                       1.616672e+09                      2021-03-25 19:30:00+08:00              1.049790e+00                                7.0                        1616671800                     2021-03-25 19:30:00+08:00                    53.50                              39.0                            <NA>                                         NaT                    NaN                             NaN   
7      MG02                                NaN                                            NaT                       NaN                                NaN                        1616671800                     2021-03-25 19:30:00+08:00                    75.00                              39.0                            <NA>                                         NaT                    NaN                             NaN   
8  apache01                                NaN                                            NaT                       NaN                                NaN                              <NA>                                           NaT                      NaN                               NaN                      1616671920                   2021-03-25 19:32:00+08:00              48.642857                             1.0   
9  apache02                                NaN                                            NaT                       NaN                                NaN                              <NA>                                           NaT                      NaN                               NaN                      1616672220                   2021-03-25 19:37:00+08:00              68.300000                             1.0   

   earliest_any_anomaly_ts earliest_any_anomaly_datetime_asia_shanghai  
0             1616671800.0                   2021-03-25 19:30:00+08:00  
1             1616671800.0                   2021-03-25 19:30:00+08:00  
2             1616671800.0                   2021-03-25 19:30:00+08:00  
3             1616671800.0                   2021-03-25 19:30:00+08:00  
4             1616671800.0                   2021-03-25 19:30:00+08:00  
5             1616671800.0                   2021-03-25 19:30:00+08:00  
6             1616671800.0                   2021-03-25 19:30:00+08:00  
7             1616671800.0                   2021-03-25 19:30:00+08:00  
8             1616671920.0                   2021-03-25 19:32:00+08:00  
9             1616672220.0                   2021-03-25 19:37:00+08:00```
```

In [ ]:
# In[9]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Validate required kernel variables
for v in ('trace_faults','anomalies_trace','anomalies_metric','selected_metric_anoms','anomalies_log'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# Work on copies
tf = trace_faults.copy()
am = anomalies_metric.copy()
sam = selected_metric_anoms.copy()
al = anomalies_log.copy()

# Ensure expected columns exist in trace_faults
expected_tf_cols = ['cmdb_id','trace_name','segment_start_ts','segment_min_value','segment_max_value','segment_max_deviation','segment_length_minutes']
for c in expected_tf_cols:
    if c not in tf.columns:
        # try alternative names from earlier steps
        if c == 'segment_start_ts' and 'segment_start_ts' not in tf.columns and 'segment_start_ts' in tf.columns:
            pass
        else:
            # missing column -> raise
            raise RuntimeError(f"trace_faults missing expected column: {c}")

# 1) Extract downstream component from trace_name using pattern 'to_<Component>'
def extract_downstream(trace_name):
    # look for 'to_<Component>' pattern
    m = re.search(r'\bto_([A-Za-z0-9]+)\b', str(trace_name))
    if m:
        return m.group(1)
    # also try 'to-<Component>' as fallback
    m2 = re.search(r'\bto-([A-Za-z0-9]+)\b', str(trace_name))
    return m2.group(1) if m2 else None

tf['downstream_cmdb_id'] = tf['trace_name'].apply(extract_downstream)
# Keep only rows that map to a downstream component
tf_down = tf[tf['downstream_cmdb_id'].notna()].copy()

# Prepare display datetime
tf_down['earliest_fault_start_timestamp'] = tf_down['segment_start_ts'].astype(int)
tf_down['earliest_fault_start_datetime_asia_shanghai'] = pd.to_datetime(tf_down['earliest_fault_start_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Select columns for downstream mapping rows
down_rows = tf_down[[
    'cmdb_id','downstream_cmdb_id','trace_name','earliest_fault_start_timestamp','earliest_fault_start_datetime_asia_shanghai',
    'segment_length_minutes','segment_max_value','segment_min_value','segment_max_deviation'
]].copy()

down_rows = down_rows.rename(columns={'cmdb_id':'upstream_cmdb_id'})

# 2) For each downstream_cmdb_id check metric anomalies (prefer selected_metric_anoms then anomalies_metric)
# Prepare metric anomaly summaries per component
def metric_summary_for_components(metric_df):
    if metric_df.empty:
        return pd.DataFrame(columns=['cmdb_id','metric_earliest_ts','metric_earliest_dt','metric_max_severity','metric_group_of_max_sev'])
    df = metric_df.copy()
    # ensure numeric earliest ts
    if 'earliest_anomaly_ts' in df.columns:
        df['earliest_anomaly_ts'] = pd.to_numeric(df['earliest_anomaly_ts'], errors='coerce')
    # compute per-component earliest ts and max severity and kpi_name of max severity
    agg_earliest = df.groupby('cmdb_id', dropna=False).agg(
        metric_earliest_ts = ('earliest_anomaly_ts','min'),
        metric_max_severity = ('severity_score','max')
    ).reset_index()
    # find kpi_name with max severity per component
    idx = df.groupby('cmdb_id')['severity_score'].idxmax().dropna()
    max_kpis = df.loc[idx, ['cmdb_id','kpi_name','severity_score']].rename(columns={'kpi_name':'metric_group_of_max_sev','severity_score':'metric_max_severity_confirm'})
    result = agg_earliest.merge(max_kpis[['cmdb_id','metric_group_of_max_sev']], on='cmdb_id', how='left')
    result['metric_earliest_dt'] = result['metric_earliest_ts'].apply(lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz) if pd.notna(e) else pd.NaT)
    return result

# Use selected_metric_anoms first, fallback to anomalies_metric for components not present
metric_sam = metric_summary_for_components(sam)
metric_all = metric_summary_for_components(am)

# Merge summaries preferring selected_metric_anoms
metric_summary = metric_all.set_index('cmdb_id').combine_first(metric_sam.set_index('cmdb_id')).reset_index()
# However we want to prefer selected_metric_anoms if present: override with sam entries
metric_summary = metric_all.copy()
if not sam.empty:
    # update rows present in sam
    for _, r in metric_sam.iterrows():
        metric_summary.loc[metric_summary['cmdb_id']==r['cmdb_id'], ['metric_earliest_ts','metric_max_severity','metric_group_of_max_sev','metric_earliest_dt']] = \
            r[['metric_earliest_ts','metric_max_severity','metric_group_of_max_sev','metric_earliest_dt']].values
# If metric_summary lacks some components but sam has them, append
missing_in_metric = set(sam['cmdb_id'].unique()) - set(metric_summary['cmdb_id'].unique())
if missing_in_metric:
    metric_summary = pd.concat([metric_summary, metric_sam[metric_sam['cmdb_id'].isin(missing_in_metric)]], ignore_index=True, sort=False)

# 3) Prepare log anomaly summary per component from anomalies_log
if not al.empty:
    log_agg = al.groupby('cmdb_id', dropna=False).agg(
        log_earliest_ts = ('earliest_anomaly_ts','min'),
        log_max_severity = ('severity_score','max')
    ).reset_index()
    # find log_name of max severity per component
    idxl = al.groupby('cmdb_id')['severity_score'].idxmax().dropna()
    max_logs = al.loc[idxl, ['cmdb_id','log_name','severity_score']].rename(columns={'log_name':'log_group_of_max_sev','severity_score':'log_max_severity_confirm'})
    log_summary = log_agg.merge(max_logs[['cmdb_id','log_group_of_max_sev']], on='cmdb_id', how='left')
    log_summary['log_earliest_dt'] = log_summary['log_earliest_ts'].apply(lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz) if pd.notna(e) else pd.NaT)
else:
    log_summary = pd.DataFrame(columns=['cmdb_id','log_earliest_ts','log_max_severity','log_group_of_max_sev','log_earliest_dt'])

# 4) Join downstream info into down_rows
# Left join metric_summary and log_summary on downstream_cmdb_id
merged = down_rows.merge(metric_summary.add_prefix('down_').rename(columns={'down_cmdb_id':'downstream_cmdb_id'}), left_on='downstream_cmdb_id', right_on='downstream_cmdb_id', how='left')
merged = merged.merge(log_summary.add_prefix('down_').rename(columns={'down_cmdb_id':'downstream_cmdb_id'}), left_on='downstream_cmdb_id', right_on='downstream_cmdb_id', how='left')

# Normalize column names for output
out = merged.rename(columns={
    'downstream_cmdb_id':'downstream_cmdb_id',
    'upstream_cmdb_id':'upstream_cmdb_id'
})

# Determine boolean flags for downstream anomalies presence
out['downstream_metric_anomaly'] = out['down_metric_earliest_ts'].notna()
out['downstream_log_anomaly'] = out['down_log_earliest_ts'].notna()

# Keep only rows where downstream_metric_anomaly OR downstream_log_anomaly is True
out_filtered = out[(out['downstream_metric_anomaly']) | (out['downstream_log_anomaly'])].copy()

# Select and order requested columns
final_cols = [
    'upstream_cmdb_id','downstream_cmdb_id','trace_name','earliest_fault_start_timestamp','earliest_fault_start_datetime_asia_shanghai',
    'segment_length_minutes','segment_max_value','segment_min_value','segment_max_deviation',
    'downstream_metric_anomaly','down_metric_earliest_ts','down_metric_earliest_dt','down_metric_max_severity','metric_group_of_max_sev',
    'downstream_log_anomaly','down_log_earliest_ts','down_log_earliest_dt','down_log_max_severity','log_group_of_max_sev'
]
# Some columns may not exist due to naming variations; filter present ones
final_cols = [c for c in final_cols if c in out_filtered.columns]
downstream_correlation = out_filtered[final_cols].copy()

# Sort by earliest_fault_start_timestamp ascending and limit to 20 rows
downstream_correlation = downstream_correlation.sort_values('earliest_fault_start_timestamp', ascending=True).reset_index(drop=True).head(20)

# Round numeric severities/deviation for compactness
for c in ['segment_max_deviation','down_metric_max_severity','down_log_max_severity']:
    if c in downstream_correlation.columns:
        downstream_correlation[c] = pd.to_numeric(downstream_correlation[c], errors='coerce').round(6)

# 5) Compute top candidate: minimal earliest_fault_start_timestamp among filtered rows
top_candidate = None
if not downstream_correlation.empty:
    # find minimal timestamp
    min_ts = downstream_correlation['earliest_fault_start_timestamp'].min()
    candidates_min = downstream_correlation[downstream_correlation['earliest_fault_start_timestamp']==min_ts].copy()
    # choose by downstream metric severity (max), fallback to log severity
    def pick_score(row):
        m = row.get('down_metric_max_severity', pd.NA)
        l = row.get('down_log_max_severity', pd.NA)
        # prefer metric severity if present
        if pd.notna(m):
            return float(m)
        if pd.notna(l):
            return float(l)
        return -1.0
    candidates_min['pick_score'] = candidates_min.apply(pick_score, axis=1)
    chosen = candidates_min.sort_values('pick_score', ascending=False).iloc[0]

    # Build reason_candidates string
    reasons = []
    if pd.notna(chosen.get('metric_group_of_max_sev')) and pd.notna(chosen.get('down_metric_max_severity')):
        reasons.append(f"metric: {chosen['metric_group_of_max_sev']} severity={float(chosen['down_metric_max_severity']):.6g}")
    if pd.notna(chosen.get('log_group_of_max_sev')) and pd.notna(chosen.get('down_log_max_severity')):
        reasons.append(f"log: {chosen['log_group_of_max_sev']} severity={float(chosen['down_log_max_severity']):.6g}")
    reason_candidates = "; ".join(reasons) if reasons else "metric/log anomaly"

    top_candidate = pd.DataFrame([{
        'downstream_cmdb_id': chosen.get('downstream_cmdb_id'),
        'upstream_cmdb_id': chosen.get('upstream_cmdb_id'),
        'earliest_fault_start_timestamp': int(chosen['earliest_fault_start_timestamp']),
        'earliest_fault_start_datetime_asia_shanghai': pd.to_datetime(int(chosen['earliest_fault_start_timestamp']), unit='s', utc=True).tz_convert(tz),
        'reason_candidates': reason_candidates
    }])
else:
    top_candidate = pd.DataFrame(columns=['downstream_cmdb_id','upstream_cmdb_id','earliest_fault_start_timestamp','earliest_fault_start_datetime_asia_shanghai','reason_candidates'])

# Store results in kernel variables
downstream_correlation = downstream_correlation.reset_index(drop=True)
top_root_candidate = top_candidate

# Final displayed variables (compact)
downstream_correlation, top_root_candidate

```
Out[9]:
```
Summary of downstream correlation (based on trace_faults merged with metric/log anomalies):

- Matches found: multiple trace fault segments point to MG02 as the downstream service (upstreams: Tomcat03, Tomcat02, IG01). These segments start mainly around 2021-03-25 19:37:00+08 and later (several single-minute segments and a few 2-minute segments).  
- Downstream metric evidence: MG02 shows metric anomalies for many groups (computed downstream metric max severity = sentinel 1e18 for some metrics due to zero min values in-window). That strong sentinel indicates either true zero I/O readings or a collection/agent issue for MG02—investigate disk/agent health on MG02.  
- Downstream log evidence: Tomcat02 (as a downstream in some traces) shows strong log anomalies (log.row_count severity ≈ 145.5), indicating large log/traffic spikes at ~19:37; several trace edges (IG01 → Tomcat02 and others) also map to Tomcat02 where both metric and log anomalies exist.  
- Filtered rows (examples): 
  - Tomcat03 → MG02, trace.to_MG02.row_count, start=2021-03-25 19:37:00+08, seg_len=1 min, seg_min/max ≈68, deviation≈1.103, downstream_metric_anomaly=True (MG02 metric earliest at 19:30), downstream_log_anomaly=False.  
  - Tomcat02 → MG02 and Tomcat03 → MG02 have repeated segments at 19:37–19:53 and at 20:00 pointing to MG02 with similar downstream metric anomalies.  
  - IG01 → Tomcat02, trace.to_Tomcat02.row_count, earliest at 19:38 and later at 19:52/20:00; Tomcat02 shows both downstream_metric_anomaly=True and downstream_log_anomaly=True (log severity ≈145.5) — i.e., correlated evidence.  
- Top candidate (selected): downstream_cmdb_id = MG02, upstream_cmdb_id = Tomcat03, earliest_fault_start_timestamp = 1616672220 → 2021-03-25 19:37:00+08. Reason: downstream (MG02) has metric anomalies (sentinel/severe) and this is the earliest downstream-correlated fault among rows with downstream metric/log evidence.

Actionable next steps:
1. Immediately inspect MG02 metrics and agent: check disk I/O, disk space, and metrics ingestion around 19:30–19:40 for zeros or agent errors.  
2. Correlate MG02 investigation with the upstream Tomcat nodes (Tomcat03, Tomcat02) at ~19:37 to see if upstream spikes/requests were routed to MG02 and caused resource issues.  
3. Investigate Tomcat02 logs/requests (high log.row_count severity ≈145.5 at 19:37) and trace spikes (IG01→Tomcat02) for request surges or errors.  
4. Use the stored tables downstream_correlation and top_root_candidate for detailed per-row inspection and share with service owners (MG02, Tomcat02/03) for root-cause follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(   upstream_cmdb_id downstream_cmdb_id                   trace_name  earliest_fault_start_timestamp earliest_fault_start_datetime_asia_shanghai  segment_length_minutes  segment_max_value  segment_min_value  segment_max_deviation  downstream_metric_anomaly  down_metric_earliest_ts   down_metric_earliest_dt  down_metric_max_severity  downstream_log_anomaly  down_log_earliest_ts      down_log_earliest_dt  \
0          Tomcat03               MG02      trace.to_MG02.row_count                      1616672220                   2021-03-25 19:37:00+08:00                       1               68.0               68.0               1.102941                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
1          Tomcat02               MG02      trace.to_MG02.row_count                      1616672220                   2021-03-25 19:37:00+08:00                       1               73.0               73.0               1.013699                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
2              IG01           Tomcat02  trace.to_Tomcat02.row_count                      1616672280                   2021-03-25 19:38:00+08:00                       1               72.0               72.0               1.013889                       True               1616671800 2021-03-25 19:30:00+08:00              1.000100e+00                    True            1616672220 2021-03-25 19:37:00+08:00   
3          Tomcat02               MG02      trace.to_MG02.row_count                      1616672640                   2021-03-25 19:44:00+08:00                       1              131.0              131.0               1.039683                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
4          Tomcat02               MG02      trace.to_MG02.row_count                      1616672880                   2021-03-25 19:48:00+08:00                       1              127.0              127.0               1.007937                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
5              IG01           Tomcat02  trace.to_Tomcat02.row_count                      1616673120                   2021-03-25 19:52:00+08:00                       1              137.0              137.0               1.078740                       True               1616671800 2021-03-25 19:30:00+08:00              1.000100e+00                    True            1616672220 2021-03-25 19:37:00+08:00   
6          Tomcat03               MG02      trace.to_MG02.row_count                      1616673180                   2021-03-25 19:53:00+08:00                       2              139.0              137.0               1.022059                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
7              IG01           Tomcat02  trace.to_Tomcat02.row_count                      1616673240                   2021-03-25 19:54:00+08:00                       1              131.0              131.0               1.031496                       True               1616671800 2021-03-25 19:30:00+08:00              1.000100e+00                    True            1616672220 2021-03-25 19:37:00+08:00   
8          Tomcat02               MG02      trace.to_MG02.row_count                      1616673600                   2021-03-25 20:00:00+08:00                       1                1.0                1.0              74.000000                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
9          Tomcat03               MG02      trace.to_MG02.row_count                      1616673600                   2021-03-25 20:00:00+08:00                       1                1.0                1.0              75.000000                       True               1616671800 2021-03-25 19:30:00+08:00              1.000000e+18                   False                  <NA>                       NaT   
10             IG01           Tomcat02  trace.to_Tomcat02.row_count                      1616673600                   2021-03-25 20:00:00+08:00                       1                1.0                1.0              73.000000                       True               1616671800 2021-03-25 19:30:00+08:00              1.000100e+00                    True            1616672220 2021-03-25 19:37:00+08:00   

    down_log_max_severity  
0                     NaN  
1                     NaN  
2                   145.5  
3                     NaN  
4                     NaN  
5                   145.5  
6                     NaN  
7                   145.5  
8                     NaN  
9                     NaN  
10                  145.5  ,   downstream_cmdb_id upstream_cmdb_id  earliest_fault_start_timestamp earliest_fault_start_datetime_asia_shanghai   reason_candidates
0               MG02         Tomcat03                      1616672220                   2021-03-25 19:37:00+08:00  metric/log anomaly)```
```

In [ ]:
# In[10]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Reuse kernel variables: selected_metric_anoms, anomalies_metric, df_metric, trace_faults
# Validate presence
for v in ('df_metric','anomalies_metric','selected_metric_anoms','trace_faults'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel.")

# 1) MG02 metric anomalies
# Prefer selected_metric_anoms (already filtered to candidates) but include any anomalies_metric entries for MG02
sam = selected_metric_anoms.copy()
am = anomalies_metric.copy()
dfm = df_metric.copy()

# Normalize column names if needed
if 'kpi_name' not in am.columns and 'metric_name' in am.columns:
    am = am.rename(columns={'metric_name':'kpi_name'})
if 'kpi_name' not in sam.columns and 'metric_name' in sam.columns:
    sam = sam.rename(columns={'metric_name':'kpi_name'})
# Ensure earliest_anomaly_ts exists and is int
for df in (sam, am):
    if 'earliest_anomaly_ts' in df.columns:
        df['earliest_anomaly_ts'] = pd.to_numeric(df['earliest_anomaly_ts'], errors='coerce').astype('Int64')
    else:
        # try datetime column
        if 'earliest_anomaly_datetime_asia_shanghai' in df.columns:
            df['earliest_anomaly_ts'] = pd.to_datetime(df['earliest_anomaly_datetime_asia_shanghai']).apply(
                lambda d: int(d.tz_convert('UTC').timestamp()) if pd.notna(d) else pd.NA
            ).astype('Int64')

# Filter MG02 from both sources
sam_mg = sam[sam['cmdb_id']=='MG02'].copy()
am_mg  = am[am['cmdb_id']=='MG02'].copy()

# Combine, preferring rows from sam_mg (sam likely subset), avoid duplicates by kpi_name
if not sam_mg.empty:
    # mark source
    sam_mg['_src'] = 'selected'
else:
    sam_mg['_src'] = []

if not am_mg.empty:
    am_mg['_src'] = 'all'
else:
    am_mg['_src'] = []

combined = pd.concat([sam_mg, am_mg], ignore_index=True, sort=False)
# Keep first occurrence per (cmdb_id,kpi_name) preferring selected ('selected' first)
combined['_pref'] = combined['_src'].map({'selected':0,'all':1}).fillna(1)
combined = combined.sort_values(['kpi_name','_pref']).drop_duplicates(subset=['kpi_name'], keep='first').reset_index(drop=True)
combined = combined.drop(columns=['_src','_pref'], errors='ignore')

# Ensure numeric fields
for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score','count_in_window']:
    if c in combined.columns:
        combined[c] = pd.to_numeric(combined[c], errors='coerce')

# Earliest anomaly datetime in Asia/Shanghai
def epoch_to_as_tz(e):
    if pd.isna(e):
        return pd.NaT
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz)

if 'earliest_anomaly_ts' in combined.columns:
    combined['earliest_anomaly_ts'] = combined['earliest_anomaly_ts'].astype('Int64')
    combined['earliest_anomaly_datetime_asia_shanghai'] = combined['earliest_anomaly_ts'].apply(lambda e: epoch_to_as_tz(e) if not pd.isna(e) else pd.NaT)
else:
    combined['earliest_anomaly_datetime_asia_shanghai'] = pd.NaT
    combined['earliest_anomaly_ts'] = pd.NA

# Retrieve earliest_sample_value by joining df_metric on (cmdb_id,kpi_name,timestamp)
# Ensure df_metric has kpi_name column and timestamp numeric
dfm['timestamp'] = pd.to_numeric(dfm['timestamp'], errors='coerce').astype('Int64')
dfm['value'] = pd.to_numeric(dfm['value'], errors='coerce')

# Prepare a key to merge
if 'earliest_anomaly_ts' in combined.columns:
    merged_samples = combined.merge(
        dfm[['cmdb_id','kpi_name','timestamp','value']].rename(columns={'timestamp':'earliest_anomaly_ts','value':'earliest_sample_value'}),
        on=['cmdb_id','kpi_name','earliest_anomaly_ts'],
        how='left'
    )
else:
    merged_samples = combined.copy()
    merged_samples['earliest_sample_value'] = pd.NA

# Final mg02_metric_anoms: select requested columns and limit to 20, sorted by earliest_anomaly_ts ascending
cols_req = [
    'cmdb_id','kpi_name','count_in_window','earliest_anomaly_ts','earliest_anomaly_datetime_asia_shanghai',
    'min_value_in_window','max_value_in_window','p95','p50','p5','severity_score','earliest_sample_value'
]
available_cols = [c for c in cols_req if c in merged_samples.columns]
mg02_metric_anoms = merged_samples[available_cols].copy()
# Sort and limit
if 'earliest_anomaly_ts' in mg02_metric_anoms.columns:
    mg02_metric_anoms = mg02_metric_anoms.sort_values('earliest_anomaly_ts', ascending=True, na_position='last').head(20).reset_index(drop=True)
else:
    mg02_metric_anoms = mg02_metric_anoms.head(20).reset_index(drop=True)

# Round numeric columns for compactness
for c in ['min_value_in_window','max_value_in_window','p95','p50','p5','severity_score','earliest_sample_value']:
    if c in mg02_metric_anoms.columns:
        mg02_metric_anoms[c] = pd.to_numeric(mg02_metric_anoms[c], errors='coerce').round(6)

# Store in kernel variable
mg02_metric_anoms = mg02_metric_anoms

# 2) MG02 downstream traces from trace_faults
tf = trace_faults.copy()

# Extract downstream component as done earlier
def extract_downstream(trace_name):
    m = re.search(r'\bto_([A-Za-z0-9]+)\b', str(trace_name))
    if m:
        return m.group(1)
    m2 = re.search(r'\bto-([A-Za-z0-9]+)\b', str(trace_name))
    return m2.group(1) if m2 else None

tf['downstream_cmdb_id'] = tf['trace_name'].apply(extract_downstream)
# Filter rows where downstream == 'MG02'
mg_tf = tf[tf['downstream_cmdb_id']=='MG02'].copy()

# Ensure segment start ts field exists; support both 'segment_start_ts' and 'segment_start_ts' naming
if 'segment_start_ts' not in mg_tf.columns and 'segment_start_ts' in mg_tf.columns:
    mg_tf['segment_start_ts'] = mg_tf['segment_start_ts']

# Prepare the requested columns and convert datetime
mg_tf['earliest_fault_start_timestamp'] = mg_tf['segment_start_ts'].astype(int)
mg_tf['earliest_fault_start_datetime_asia_shanghai'] = pd.to_datetime(mg_tf['earliest_fault_start_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Select and rename columns for output
cols_tf = [
    ('cmdb_id','upstream_cmdb_id'),
    ('downstream_cmdb_id','downstream_cmdb_id'),
    ('trace_name','trace_name'),
    ('earliest_fault_start_timestamp','earliest_fault_start_timestamp'),
    ('earliest_fault_start_datetime_asia_shanghai','earliest_fault_start_datetime_asia_shanghai'),
    ('segment_length_minutes','segment_length_minutes'),
    ('segment_min_value','segment_min_value'),
    ('segment_max_value','segment_max_value'),
    ('segment_max_deviation','segment_max_deviation')
]

available_tf_cols = [(a,b) for a,b in cols_tf if a in mg_tf.columns]
mg02_downstream_traces = mg_tf[[a for a,b in available_tf_cols]].copy()
# Rename to requested output names
rename_map = {a:b for a,b in available_tf_cols}
mg02_downstream_traces = mg02_downstream_traces.rename(columns=rename_map)

# Sort by earliest_fault_start_timestamp ascending and limit to 20
if 'earliest_fault_start_timestamp' in mg02_downstream_traces.columns:
    mg02_downstream_traces = mg02_downstream_traces.sort_values('earliest_fault_start_timestamp', ascending=True).head(20).reset_index(drop=True)
else:
    mg02_downstream_traces = mg02_downstream_traces.head(20).reset_index(drop=True)

# Round numeric columns for compactness
for c in ['segment_min_value','segment_max_value','segment_max_deviation']:
    if c in mg02_downstream_traces.columns:
        mg02_downstream_traces[c] = pd.to_numeric(mg02_downstream_traces[c], errors='coerce').round(6)

# Store results in kernel variables (already assigned)
mg02_metric_anoms, mg02_downstream_traces

```
Out[10]:
```
```python
# Concise plain-English summary of MG02 evidence based on stored tables mg02_metric_anoms and mg02_downstream_traces.
summary = (
    "MG02 evidence summary:\n\n"
    "1) Metric anomalies for MG02 (stored in mg02_metric_anoms):\n"
    "- Many MG02 metrics show anomalies with earliest anomaly at 2021-03-25 19:30:00+08.\n"
    "- Affected metric types include filesystem capacities/inode usage, memory (FreeMem, CacheMem, UserMem), JVM uptime, CPU (idle/util/load), network (TotalTcpConnNum, NETKBTotalPerSec), and several local-disk metrics.\n"
    "- Notably, some local-disk series (e.g., LOCALDISK sdb DSKRTps and DSKRead) have min_value_in_window == 0 which produced a sentinel severity (1e18) due to safe division handling — this suggests either true zero I/O or an agent/collection issue for disk metrics.\n"
    "- Other metric deviations are modest (>1 but close to 1) except where sentinel appears.\n\n"
    "2) Downstream trace evidence pointing to MG02 (stored in mg02_downstream_traces):\n"
    "- Several upstream services link to MG02 via traces 'trace.to_MG02.row_count' (upstreams include Tomcat03 and Tomcat02, and some IG->Tomcat->MG edges).\n"
    "- Earliest downstream-correlated trace faults start at 2021-03-25 19:37:00+08 (Tomcat03->MG02 and Tomcat02->MG02), with additional segments later (19:44, 19:48, 19:53, 20:00). Most segments are single-minute; one segment spans 2 minutes.\n"
    "- Segment values (row_count) are in the tens to low hundreds (e.g., 68–139) with moderate deviations (segment_max_deviation ~1.01–1.10). Some late single-minute segments show very small values producing large deviation numbers due to threshold math (handled earlier).\n\n"
    "3) Correlation and interpretation:\n"
    "- MG02 has clear metric anomalies coincident with upstream trace faults pointing to it. The combination of trace evidence (many upstreams hitting MG02 around ~19:37) and metric anomalies (including disk/network and CPU/memory) makes MG02 a strong candidate for further investigation.\n"
    "- The sentinel severity for certain disk metrics (zeros) is important: investigate whether zeros are valid (no I/O) or caused by metric collection failures.\n\n"
    "4) Recommended next steps:\n"
    "- Immediately check MG02: disk I/O, disk health, disk capacity, filesystem agents, and metric collector logs around 19:30–19:40.\n"
    "- Correlate MG02 metrics with upstream Tomcat nodes (Tomcat02, Tomcat03) around 19:37 to see if request spikes caused MG02 resource issues.\n"
    "- Inspect Tomcat logs and GC around the same times (trace and log anomalies were observed for Tomcat nodes).\n"
    "\n"
    "Stored artifacts: full per-metric rows are in kernel variable 'mg02_metric_anoms'; downstream trace segments are in 'mg02_downstream_traces'."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  count_in_window  earliest_anomaly_ts earliest_anomaly_datetime_asia_shanghai  min_value_in_window  max_value_in_window           p95           p50            p5  severity_score
0     MG02  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...               19           1616671800               2021-03-25 19:30:00+08:00         8.334000e-01         8.344000e-01  8.344000e-01  8.341000e-01  8.336700e-01    1.000324e+00
1     MG02  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...                9           1616671800               2021-03-25 19:30:00+08:00         2.236000e-01         2.264000e-01  2.262400e-01  2.252000e-01  2.237600e-01    1.000716e+00
2     MG02      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity                9           1616671800               2021-03-25 19:30:00+08:00         7.264310e+01         7.362580e+01  7.357188e+01  7.320020e+01  7.270662e+01    1.000874e+00
3     MG02        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity                9           1616671800               2021-03-25 19:30:00+08:00         2.756500e+00         2.756800e+00  2.756800e+00  2.756700e+00  2.756540e+00    1.000015e+00
4     MG02  OSLinux-OSLinux_FILESYSTEM_----tomcat_FSInodeU...               15           1616671800               2021-03-25 19:30:00+08:00         2.236020e-01         2.264400e-01  2.262910e-01  2.250060e-01  2.237520e-01    1.000670e+00
5     MG02   OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity               15           1616671800               2021-03-25 19:30:00+08:00         7.264314e+01         7.362577e+01  7.357650e+01  7.313431e+01  7.270446e+01    1.000844e+00
6     MG02     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity               15           1616671800               2021-03-25 19:30:00+08:00         2.756531e+00         2.756778e+00  2.756778e+00  2.756737e+00  2.756560e+00    1.000010e+00
7     MG02         OSLinux-OSLinux_FILESYSTEM_----_FSCapacity               15           1616671800               2021-03-25 19:30:00+08:00         3.031419e+01         3.031430e+01  3.031430e+01  3.031422e+01  3.031421e+01    1.000001e+00
8     MG02           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem               18           1616671800               2021-03-25 19:30:00+08:00         1.170000e+02         1.520000e+02  1.494500e+02  1.280000e+02  1.170000e+02    1.017063e+00
9     MG02             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem               18           1616671800               2021-03-25 19:30:00+08:00         5.499000e+03         5.533000e+03  5.531300e+03  5.520000e+03  5.499850e+03    1.000307e+00
10    MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime               31           1616671800               2021-03-25 19:30:00+08:00         7.870885e+06         7.872692e+06  7.872601e+06  7.871787e+06  7.870974e+06    1.000012e+00
11    MG02              OSLinux-OSLinux_MEMORY_MEMORY_UserMem               20           1616671800               2021-03-25 19:30:00+08:00         7.636000e+03         7.672000e+03  7.671050e+03  7.660000e+03  7.639800e+03    1.000498e+00
12    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps               15           1616671800               2021-03-25 19:30:00+08:00         0.000000e+00         1.670000e-02  5.010000e-03  0.000000e+00  0.000000e+00    1.000000e+18
13    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead               15           1616671800               2021-03-25 19:30:00+08:00         0.000000e+00         1.333000e-01  3.999000e-02  0.000000e+00  0.000000e+00    1.000000e+18
14    MG02                    JVM-Runtime_7779_JVM_JVM_Uptime               21           1616671860               2021-03-25 19:31:00+08:00         2.798040e+05         2.815440e+05  2.814240e+05  2.806440e+05  2.798640e+05    1.000426e+00
15    MG02                    OSLinux-CPU_CPU-0_SingleCpuUtil               10           1616671860               2021-03-25 19:31:00+08:00         1.948100e+00         3.960700e+00  3.757210e+00  2.462150e+00  2.022035e+00    1.054160e+00
16    MG02                        OSLinux-CPU_CPU_CPUidleutil               31           1616671860               2021-03-25 19:31:00+08:00         7.229450e+01         7.350720e+01  7.343950e+01  7.321050e+01  7.252653e+01    1.003210e+00
17    MG02                            OSLinux-CPU_CPU_CPULoad               31           1616671860               2021-03-25 19:31:00+08:00         1.000000e+00         1.600000e+00  1.255000e+00  1.050000e+00  1.000000e+00    1.274900e+00
18    MG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum               21           1616671920               2021-03-25 19:32:00+08:00         1.717000e+03         2.621000e+03  2.478000e+03  2.060000e+03  1.758000e+03    1.057708e+00
19    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec               15           1616671920               2021-03-25 19:32:00+08:00         1.137080e+05         4.083670e+05  3.908978e+05  3.021660e+05  1.778973e+05    1.564510e+00,   upstream_cmdb_id downstream_cmdb_id               trace_name  earliest_fault_start_timestamp earliest_fault_start_datetime_asia_shanghai  segment_length_minutes  segment_min_value  segment_max_value  segment_max_deviation
0         Tomcat03               MG02  trace.to_MG02.row_count                      1616672220                   2021-03-25 19:37:00+08:00                       1               68.0               68.0               1.102941
1         Tomcat02               MG02  trace.to_MG02.row_count                      1616672220                   2021-03-25 19:37:00+08:00                       1               73.0               73.0               1.013699
2         Tomcat02               MG02  trace.to_MG02.row_count                      1616672640                   2021-03-25 19:44:00+08:00                       1              131.0              131.0               1.039683
3         Tomcat02               MG02  trace.to_MG02.row_count                      1616672880                   2021-03-25 19:48:00+08:00                       1              127.0              127.0               1.007937
4         Tomcat03               MG02  trace.to_MG02.row_count                      1616673180                   2021-03-25 19:53:00+08:00                       2              137.0              139.0               1.022059
5         Tomcat03               MG02  trace.to_MG02.row_count                      1616673600                   2021-03-25 20:00:00+08:00                       1                1.0                1.0              75.000000
6         Tomcat02               MG02  trace.to_MG02.row_count                      1616673600                   2021-03-25 20:00:00+08:00                       1                1.0                1.0              74.000000)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')
SENTINEL = 1e18

# Reuse existing kernel variables: df_trace, thresh_trace, df_metric, thresh_metric
# Incident window
start_ts = 1616671800
end_ts   = 1616673600

# --- 1) Trace latency check for MG02 duration series ---
# Select trace rows for MG02 within window where trace_name contains 'duration'
tf_mg = df_trace[(df_trace['cmdb_id']=='MG02') & 
                 (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts) &
                 (df_trace['trace_name'].str.contains('duration', case=False, na=False))
                ].copy()

# Group window summary
trace_group_win = (
    tf_mg
    .groupby(['trace_name'], dropna=False)
    .agg(
        count_in_window = ('value','count')
    )
    .reset_index()
)

# Prepare global thresholds for MG02 traces from thresh_trace (metric_name == trace_name)
# Ensure numeric
thresh_trace['p95'] = pd.to_numeric(thresh_trace['p95'], errors='coerce')
thresh_trace['p50'] = pd.to_numeric(thresh_trace['p50'], errors='coerce')
thresh_trace['p5']  = pd.to_numeric(thresh_trace['p5'], errors='coerce')

# Filter thresholds for MG02
th_mg = thresh_trace[thresh_trace['cmdb_id']=='MG02'][['metric_name','p95','p50','p5']].rename(columns={'metric_name':'trace_name'})

# Merge group summary with thresholds
trace_checks = trace_group_win.merge(th_mg, on='trace_name', how='left')

# For each trace_name, find earliest timestamp in-window where value > p95
def safe_first_exceed(group):
    p95 = group.name_p95 if 'name_p95' in group.attrs else None
    # group is series rows. We'll just return min timestamp where value>p95
    return None

# Row-level merge to find earliest exceed
tf_mg['value'] = pd.to_numeric(tf_mg['value'], errors='coerce')
# Merge p95 into rows
tf_rows = tf_mg.merge(th_mg, on='trace_name', how='left')

# Flag exceed rows
tf_rows['exceeds_p95'] = tf_rows['value'] > tf_rows['p95']

# Earliest exceed per trace_name
earliest_exceed = (
    tf_rows[tf_rows['exceeds_p95']]
    .groupby('trace_name', dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_exceed_ts'})
)

# Value at earliest_exceed_ts
if not earliest_exceed.empty:
    earliest_exceed = earliest_exceed.merge(
        tf_rows[['trace_name','timestamp','value']],
        left_on=['trace_name','earliest_exceed_ts'],
        right_on=['trace_name','timestamp'],
        how='left'
    ).rename(columns={'value':'value_at_earliest_exceed'}).drop(columns=['timestamp'])
else:
    earliest_exceed['value_at_earliest_exceed'] = pd.Series(dtype='float')

# Combine into trace_checks
trace_checks = trace_checks.merge(earliest_exceed, on='trace_name', how='left')

# Add boolean exceeded_p95
trace_checks['exceeded_p95'] = trace_checks['earliest_exceed_ts'].notna()

# Convert earliest_exceed_ts to datetime Asia/Shanghai
trace_checks['earliest_exceed_datetime_asia_shanghai'] = trace_checks['earliest_exceed_ts'].apply(
    lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz) if not pd.isna(e) else pd.NaT
)

# Select and order columns, limit to 10 rows sorted by earliest_exceed_ts ascending (NaT last)
trace_checks_disp = trace_checks[[
    'trace_name','count_in_window','exceeded_p95','earliest_exceed_ts','earliest_exceed_datetime_asia_shanghai',
    'value_at_earliest_exceed','p95','p50','p5'
]].copy()

trace_checks_disp = trace_checks_disp.sort_values('earliest_exceed_ts', na_position='last').head(10).reset_index(drop=True)

# Round numeric columns for compactness
for c in ['value_at_earliest_exceed','p95','p50','p5']:
    if c in trace_checks_disp.columns:
        trace_checks_disp[c] = pd.to_numeric(trace_checks_disp[c], errors='coerce').round(6)

mg02_trace_latency_check = trace_checks_disp

# --- 2) MG02 network/connection metric check ---
# Identify MG02 metrics of interest from thresh_metric (global thresholds precomputed)
patterns = ['NETWORK','TotalTcpConn','NETKB']
mask = thresh_metric['cmdb_id'].eq('MG02') & thresh_metric['metric_name'].str.contains('|'.join(patterns), case=False, na=False)
net_metrics = thresh_metric[mask].copy()

# Ensure numeric thresholds
for c in ['p95','p50','p5']:
    net_metrics[c] = pd.to_numeric(net_metrics[c], errors='coerce')

# For each metric, analyze window rows
dfm = df_metric.copy()
dfm['value'] = pd.to_numeric(dfm['value'], errors='coerce')
dfm_window = dfm[(dfm['cmdb_id']=='MG02') & (dfm['timestamp']>=start_ts) & (dfm['timestamp']<=end_ts)].copy()

rows = []
for _, r in net_metrics.iterrows():
    kname = r['metric_name']
    p95 = r['p95']
    p50 = r['p50']
    p5  = r['p5']
    grp = dfm_window[dfm_window['kpi_name']==kname].copy()
    count_in_window = int(grp['value'].count())
    if count_in_window == 0:
        # no samples in window; skip or report NA with count 0; include as no exceed
        rows.append({
            'metric_name': kname,
            'count_in_window': 0,
            'earliest_ts_exceed': pd.NA,
            'earliest_datetime_asia_shanghai': pd.NaT,
            'value_at_ts': pd.NA,
            'p95': p95,
            'p50': p50,
            'p5': p5,
            'severity': pd.NA
        })
        continue
    grp = grp.sort_values('timestamp')
    # find earliest exceed where value > p95
    exceed = grp[grp['value'] > p95]
    if not exceed.empty:
        et = int(exceed['timestamp'].min())
        val = float(exceed[exceed['timestamp']==et]['value'].iloc[0])
        sev = (val / p95) if (p95 not in (0, None, np.nan)) else SENTINEL
        rows.append({
            'metric_name': kname,
            'count_in_window': count_in_window,
            'earliest_ts_exceed': et,
            'earliest_datetime_asia_shanghai': pd.to_datetime(et, unit='s', utc=True).tz_convert(tz),
            'value_at_ts': round(val,6),
            'p95': round(p95,6) if pd.notna(p95) else pd.NA,
            'p50': round(p50,6) if pd.notna(p50) else pd.NA,
            'p5': round(p5,6) if pd.notna(p5) else pd.NA,
            'severity': round(sev,6) if pd.notna(sev) else pd.NA
        })
    else:
        # no exceed: report earliest timestamp where value is maximal within window
        max_val = grp['value'].max()
        # find earliest ts where value == max_val
        et = int(grp[grp['value']==max_val]['timestamp'].min())
        val = float(max_val) if pd.notna(max_val) else np.nan
        sev = (val / p95) if (p95 not in (0, None, np.nan)) else SENTINEL
        rows.append({
            'metric_name': kname,
            'count_in_window': count_in_window,
            'earliest_ts_exceed': et,
            'earliest_datetime_asia_shanghai': pd.to_datetime(et, unit='s', utc=True).tz_convert(tz),
            'value_at_ts': round(val,6),
            'p95': round(p95,6) if pd.notna(p95) else pd.NA,
            'p50': round(p50,6) if pd.notna(p50) else pd.NA,
            'p5': round(p5,6) if pd.notna(p5) else pd.NA,
            'severity': round(sev,6) if pd.notna(sev) else pd.NA
        })

mg02_network_check = pd.DataFrame(rows)

# Sort by earliest_ts_exceed ascending and limit to 10
if 'earliest_ts_exceed' in mg02_network_check.columns:
    mg02_network_check = mg02_network_check.sort_values('earliest_ts_exceed', na_position='last').head(10).reset_index(drop=True)

# Final displayed variables
mg02_trace_latency_check, mg02_network_check

```
Out[11]:
```
Summary of MG02 checks (window 2021-03-25 19:30–20:00 +08):

1) Trace latency check (trace.duration series)
- Multiple MG02-related trace duration series exceeded their global P95 within the window. Earliest exceedances:
  - trace.from_Tomcat02.duration_p95 — earliest exceed 19:30:00, value 1.42085s (p95=1.313575)
  - trace.to_dockerB1.duration_mean — earliest exceed 19:30:00, value 0.307033s (p95=0.301208)
  - trace.to_dockerA2.duration_mean — earliest exceed 19:31:00, value 0.124898s (p95=0.10192)
  - Several other duration series (both .mean and .p95 variants, self.duration and from/to paths) show earliest exceedances at 19:36:00 (many traces) with values modestly above their P95s (examples: trace.from_Tomcat03.duration_p95 value 1.8415 > p95 1.6998; trace.self.duration_mean value 0.143827 > p95 0.135124).
- Interpretation: There is concrete trace-level latency evidence affecting MG02 (and upstream callers) starting as early as 19:30 with clusters around 19:36. The duration spikes are not enormous multiples of P95 but are clearly above expected thresholds across multiple trace paths — consistent with service latency.

2) Network / connection metric check
- Several MG02 network/connection metrics show high values within the window:
  - OSLinux-OSLinux-NETWORK-TotalTcpConnNum — earliest exceed 19:32:00, value 2621 vs p95 2478 (severity ≈ 1.0577). Indicates increased simultaneous TCP connections.
  - OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec — earliest exceed 19:32:00, value ~408,367 KB/s vs p95 ~390,898 KB/s (severity ≈ 1.0447). Indicates elevated network throughput.
  - Several network error/count metrics (NETInErr, NETOutErr, NETInErrPrc, NETOutErrPrc, TCP-CLOSE-WAIT, TCP-FIN-WAIT) show values of 0 while their p95 is 0 (or produce sentinel severity) — these produce sentinel severity by our safe-division logic; these zeros may indicate either true zero counts or collection/metric peculiarities and should be interpreted carefully.
  - Some metrics had no exceed but their maximal in-window value is at 19:32 or later (we reported the maximal timestamp when no exceed was present).
- Interpretation: MG02 experienced increased network connections and bandwidth around ~19:32, concurrent with trace latency increases. This suggests network load/stress could be contributing to the observed latency.

Overall conclusion (root-cause indication)
- Evidence supports that MG02 experienced service latency during the incident window: multiple trace duration series exceeded P95 (latency spikes) and key network metrics (connection count and bandwidth) increased around the same time. This pattern is consistent with resource pressure on MG02 (network I/O or connection saturation) causing higher request latencies.
- The sentinel severity values on some local-disk and network error counters (zeros) require validation: they could indicate real zero/error conditions or metric collection artifacts. They should not be ignored but verified.

Recommended next actions
- Investigate MG02 network subsystem around 19:30–19:36: check TCP connection counts, socket states, NIC errors, and throughput; confirm whether connection surge or bandwidth saturation occurred.
- Correlate with upstream request spikes (Tomcat02/03) at ~19:37 and MG02 disk/CPU metrics to determine whether the issue is network-limited or combined with CPU/disk pressure.
- Verify metric collection for zero-valued error counters to rule out monitoring/agent issues before concluding those are real.
- If confirmed, mitigate by throttling upstream traffic, increasing MG02 network capacity/settings, or scaling MG02 instances.

The original code execution output of IPython Kernel is also provided below for reference:

(                          trace_name  count_in_window  exceeded_p95  earliest_exceed_ts earliest_exceed_datetime_asia_shanghai  value_at_earliest_exceed       p95       p50        p5
0   trace.from_Tomcat02.duration_p95               31          True          1616671800              2021-03-25 19:30:00+08:00                  1.420850  1.313575  1.058250  0.082700
1    trace.to_dockerB1.duration_mean               31          True          1616671800              2021-03-25 19:30:00+08:00                  0.307033  0.301208  0.190387  0.042902
2    trace.to_dockerA2.duration_mean               31          True          1616671860              2021-03-25 19:31:00+08:00                  0.124898  0.101920  0.065313  0.020502
3  trace.from_Tomcat02.duration_mean               31          True          1616672160              2021-03-25 19:36:00+08:00                  0.271286  0.248932  0.144535  0.046353
4   trace.from_Tomcat04.duration_p95               31          True          1616672160              2021-03-25 19:36:00+08:00                  1.119750  1.108725  1.028100  0.072100
5           trace.self.duration_mean               31          True          1616672160              2021-03-25 19:36:00+08:00                  0.143827  0.135124  0.098270  0.054248
6   trace.from_Tomcat03.duration_p95               31          True          1616672160              2021-03-25 19:36:00+08:00                  1.841500  1.699800  1.022900  0.069000
7  trace.from_Tomcat03.duration_mean               31          True          1616672160              2021-03-25 19:36:00+08:00                  0.310242  0.249126  0.118736  0.041766
8            trace.self.duration_p95               31          True          1616672160              2021-03-25 19:36:00+08:00                  1.070900  1.042850  0.856500  0.085150
9     trace.to_dockerB1.duration_p95               31          True          1616672160              2021-03-25 19:36:00+08:00                  2.085000  2.048550  1.090200  0.072025,                                        metric_name  count_in_window  earliest_ts_exceed earliest_datetime_asia_shanghai   value_at_ts           p95          p50            p5      severity
0   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               30          1616671800       2021-03-25 19:30:00+08:00  0.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
1  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum               21          1616671920       2021-03-25 19:32:00+08:00  2.621000e+03  2.478000e+03    2060.0000    1758.00000  1.057708e+00
2     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc               10          1616671920       2021-03-25 19:32:00+08:00  0.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
3  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec               15          1616671920       2021-03-25 19:32:00+08:00  4.083670e+05  3.908978e+05  302166.0000  177897.30000  1.044690e+00
4       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc               10          1616671920       2021-03-25 19:32:00+08:00  0.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
5          OSLinux-OSLinux_NETWORK_ens160_NETInErr                9          1616671920       2021-03-25 19:32:00+08:00  0.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
6         OSLinux-OSLinux_NETWORK_ens160_NETOutErr                9          1616671920       2021-03-25 19:32:00+08:00  0.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
7     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               30          1616672040       2021-03-25 19:34:00+08:00  1.000000e+00  0.000000e+00       0.0000       0.00000  1.000000e+18
8  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil                9          1616672040       2021-03-25 19:34:00+08:00  2.800000e-03  2.680000e-03       0.0023       0.00138  1.044776e+00
9      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn               15          1616673360       2021-03-25 19:56:00+08:00  2.859932e+09  2.859827e+09     694.0000     542.80000  1.000037e+00)```
```